<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/01_Dataset_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# CELL 01.1 — DATASET ACQUISITION
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"AIR-LLM project root not found:\n{PROJECT_ROOT}"
    )

print("=" * 100)
print("AIR-LLM — NOTEBOOK 01: DATASET PREPARATION")
print("=" * 100)
print(f"Project root:\n{PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
AIR-LLM — NOTEBOOK 01: DATASET PREPARATION
Project root:
/content/drive/MyDrive/AIR_LLM_Research


In [5]:
# ============================================================
# CELL 01.2 — LOAD NOTEBOOK 00 CONFIGURATION
# ============================================================

import os
import json
import yaml
import hashlib
import warnings
from datetime import datetime, timezone

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

CONFIG_DIR = PROJECT_ROOT / "config"

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

REFERENCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "reference"
)

ARTIFACT_DIR = (
    PROJECT_ROOT
    / "artifacts"
)

for directory in [
    CONFIG_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    REFERENCE_DIR,
    ARTIFACT_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


MASTER_CONFIG_PATH = (
    CONFIG_DIR / "config.yaml"
)

DATASET_REGISTRY_PATH = (
    CONFIG_DIR / "dataset_registry.yaml"
)

FEATURE_TARGET_REGISTRY_PATH = (
    CONFIG_DIR
    / "feature_target_registry.yaml"
)

CONFIGURATION_LOCK_PATH = (
    ARTIFACT_DIR
    / "configuration_lock.json"
)


REQUIRED_CONFIG_FILES = [
    MASTER_CONFIG_PATH,
    DATASET_REGISTRY_PATH,
    FEATURE_TARGET_REGISTRY_PATH,
    CONFIGURATION_LOCK_PATH
]

missing_config_files = [
    str(path)
    for path in REQUIRED_CONFIG_FILES
    if not path.exists()
]

if missing_config_files:

    raise FileNotFoundError(
        "Required Notebook 00 configuration files are missing:\n"
        + "\n".join(missing_config_files)
    )


with open(
    MASTER_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:

    MASTER_CONFIG = yaml.safe_load(file)


with open(
    DATASET_REGISTRY_PATH,
    "r",
    encoding="utf-8"
) as file:

    DATASET_REGISTRY = yaml.safe_load(file)


with open(
    FEATURE_TARGET_REGISTRY_PATH,
    "r",
    encoding="utf-8"
) as file:

    FEATURE_TARGET_REGISTRY = yaml.safe_load(file)


with open(
    CONFIGURATION_LOCK_PATH,
    "r",
    encoding="utf-8"
) as file:

    CONFIGURATION_LOCK = json.load(file)


if CONFIGURATION_LOCK.get("status") != "LOCKED":

    raise RuntimeError(
        "Notebook 00 configuration is not LOCKED. "
        "Notebook 01 cannot continue."
    )


DATASETS = list(
    DATASET_REGISTRY.keys()
)


print("=" * 100)
print("NOTEBOOK 00 CONFIGURATION LOADED")
print("=" * 100)

print(
    f"Configuration status : "
    f"{CONFIGURATION_LOCK.get('status')}"
)

print(
    f"Project              : "
    f"{MASTER_CONFIG.get('project', {}).get('name')}"
)

print(
    f"Configuration version: "
    f"{MASTER_CONFIG.get('project', {}).get('configuration_version')}"
)

print(
    f"Datasets             : "
    f"{DATASETS}"
)

NOTEBOOK 00 CONFIGURATION LOADED
Configuration status : LOCKED
Project              : AIR-LLM Research Project
Configuration version: CONFIG-v1
Datasets             : ['adult_income', 'bank_marketing', 'diabetes_130us']


In [6]:
# ============================================================
# CELL 01.3 — RAW FILE DETECTION
# ============================================================

SUPPORTED_RAW_EXTENSIONS = {
    ".csv",
    ".tsv",
    ".txt",
    ".data"
}


def resolve_configured_raw_path(
    dataset_id,
    metadata
):

    configured_file = metadata.get(
        "raw_file"
    )

    if configured_file is None:
        return None

    configured_path = Path(
        str(configured_file)
    )

    candidates = [

        configured_path,

        PROJECT_ROOT
        / configured_path,

        RAW_DIR
        / configured_path.name
    ]

    for candidate in candidates:

        if candidate.exists() and candidate.is_file():

            return candidate.resolve()

    return None


ALL_RAW_FILES = sorted(
    [
        path
        for path in RAW_DIR.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in SUPPORTED_RAW_EXTENSIONS
        )
    ]
)


RAW_FILE_MAP = {}

RAW_DETECTION_ROWS = []


for dataset_id in DATASETS:

    metadata = DATASET_REGISTRY[
        dataset_id
    ]

    candidate = (
        resolve_configured_raw_path(
            dataset_id,
            metadata
        )
    )

    if candidate is None:

        dataset_tokens = [
            token.lower()
            for token
            in dataset_id.split("_")
        ]

        possible = [
            path
            for path in ALL_RAW_FILES
            if all(
                token in path.name.lower()
                for token in dataset_tokens
            )
        ]

        if len(possible) == 1:
            candidate = possible[0]

    RAW_FILE_MAP[
        dataset_id
    ] = candidate

    RAW_DETECTION_ROWS.append({

        "dataset_id":
            dataset_id,

        "dataset_name":
            metadata["dataset_name"],

        "raw_file_configured":
            metadata.get("raw_file"),

        "raw_file_found":
            candidate is not None,

        "raw_path":
            str(candidate)
            if candidate is not None
            else None,

        "file_name":
            candidate.name
            if candidate is not None
            else None,

        "extension":
            candidate.suffix.lower()
            if candidate is not None
            else None,

        "size_bytes":
            candidate.stat().st_size
            if candidate is not None
            else None
    })


RAW_DETECTION_DF = pd.DataFrame(
    RAW_DETECTION_ROWS
)

display(
    RAW_DETECTION_DF
)


if not RAW_DETECTION_DF[
    "raw_file_found"
].all():

    missing = (
        RAW_DETECTION_DF.loc[
            ~RAW_DETECTION_DF["raw_file_found"],
            "dataset_id"
        ].tolist()
    )

    raise FileNotFoundError(
        "Raw files not found for:\n"
        + "\n".join(missing)
        + f"\n\nExpected location:\n{RAW_DIR}"
    )


print(
    f"Detected {len(DATASETS)} / "
    f"{len(DATASETS)} datasets."
)

,dataset_id,dataset_name,raw_file_configured,raw_file_found,raw_path,file_name,extension,size_bytes
0,adult_income,Adult Income,adult_income.csv,True,/content/drive/MyDrive/AIR_LLM_Research/data/r...,adult_income.csv,.csv,3974305
1,bank_marketing,Bank Marketing,bank_marketing.csv,True,/content/drive/MyDrive/AIR_LLM_Research/data/r...,bank_marketing.csv,.csv,4610348
2,diabetes_130us,Diabetes 130-US Hospitals,diabetes_130us.csv,True,/content/drive/MyDrive/AIR_LLM_Research/data/r...,diabetes_130us.csv,.csv,16246860


Detected 3 / 3 datasets.


In [7]:
# ============================================================
# CELL 01.4 — FILE FORMAT VALIDATION
# ============================================================

def detect_file_format(path):

    extension = (
        path.suffix.lower()
    )

    if extension == ".csv":
        return "csv"

    if extension == ".tsv":
        return "tsv"

    if extension in {
        ".txt",
        ".data"
    }:
        return "delimited_text"

    return "unsupported"


FORMAT_ROWS = []


for dataset_id, path in RAW_FILE_MAP.items():

    file_format = (
        detect_file_format(path)
    )

    FORMAT_ROWS.append({

        "dataset_id":
            dataset_id,

        "file_name":
            path.name,

        "extension":
            path.suffix.lower(),

        "detected_format":
            file_format,

        "supported":
            file_format != "unsupported",

        "size_bytes":
            path.stat().st_size
    })


FILE_FORMAT_DF = pd.DataFrame(
    FORMAT_ROWS
)

display(
    FILE_FORMAT_DF
)


if not FILE_FORMAT_DF[
    "supported"
].all():

    invalid = (
        FILE_FORMAT_DF.loc[
            ~FILE_FORMAT_DF["supported"],
            "dataset_id"
        ].tolist()
    )

    raise ValueError(
        f"Unsupported raw file format: {invalid}"
    )


print(
    "File-format validation: PASSED"
)

,dataset_id,file_name,extension,detected_format,supported,size_bytes
0,adult_income,adult_income.csv,.csv,csv,True,3974305
1,bank_marketing,bank_marketing.csv,.csv,csv,True,4610348
2,diabetes_130us,diabetes_130us.csv,.csv,csv,True,16246860


File-format validation: PASSED


In [8]:
# ============================================================
# CELL 01.5 — HEADER VALIDATION
# ============================================================

HEADERLESS_DATASETS = {
    "adult_income"
}


def read_first_line(path):

    with open(
        path,
        "r",
        encoding="utf-8-sig",
        errors="replace"
    ) as file:

        return file.readline().rstrip(
            "\r\n"
        )


HEADER_ROWS = []


for dataset_id, path in RAW_FILE_MAP.items():

    first_line = read_first_line(
        path
    )

    is_known_headerless = (
        dataset_id
        in HEADERLESS_DATASETS
    )

    HEADER_ROWS.append({

        "dataset_id":
            dataset_id,

        "file_name":
            path.name,

        "header_present":
            not is_known_headerless
            and bool(first_line.strip()),

        "header_status":
            (
                "KNOWN_HEADERLESS_DATASET"
                if is_known_headerless
                else (
                    "HEADER_DETECTED"
                    if first_line.strip()
                    else "HEADER_MISSING"
                )
            ),

        "header_raw":
            first_line,

        "header_length":
            len(first_line)
    })


HEADER_VALIDATION_DF = pd.DataFrame(
    HEADER_ROWS
)

display(
    HEADER_VALIDATION_DF
)


unexpected_missing_headers = (
    HEADER_VALIDATION_DF.loc[
        (
            ~HEADER_VALIDATION_DF[
                "header_present"
            ]
        )
        &
        (
            HEADER_VALIDATION_DF[
                "header_status"
            ]
            != "KNOWN_HEADERLESS_DATASET"
        ),
        "dataset_id"
    ].tolist()
)


if unexpected_missing_headers:

    raise ValueError(
        "Unexpected missing header detected for:\n"
        + "\n".join(
            unexpected_missing_headers
        )
    )


print(
    "Header validation: PASSED"
)

,dataset_id,file_name,header_present,header_status,header_raw,header_length
0,adult_income,adult_income.csv,False,KNOWN_HEADERLESS_DATASET,"39, State-gov, 77516, Bachelors, 13, Never-mar...",127
1,bank_marketing,bank_marketing.csv,True,HEADER_DETECTED,"""age"";""job"";""marital"";""education"";""default"";""b...",150
2,diabetes_130us,diabetes_130us.csv,True,HEADER_DETECTED,"race,gender,age,weight,admission_type_id,disch...",641


Header validation: PASSED


In [38]:
# ============================================================
# CELL 01.6 — DATASET LOADING
# ============================================================
#
# Purpose:
#   Load every raw dataset using dataset-specific structural
#   information from Notebook 00.
#
# Important:
#   Adult Income is headerless and MUST be loaded with
#   header=None so that its first observation is preserved.
#
#   Bank Marketing and Diabetes 130-US are headered datasets.
# ============================================================

RAW_DATASETS = {}

DATASET_LOAD_ROWS = []


# ------------------------------------------------------------
# Canonical delimiter configuration
# ------------------------------------------------------------

DEFAULT_DELIMITERS = {
    "adult_income": ",",
    "bank_marketing": ";",
    "diabetes_130us": ","
}


# ------------------------------------------------------------
# Canonical header configuration
# ------------------------------------------------------------

HEADERLESS_DATASETS = {
    "adult_income"
}


# ------------------------------------------------------------
# Canonical Adult Income schema
# ------------------------------------------------------------

ADULT_INCOME_COLUMNS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]


# ------------------------------------------------------------
# Dataset-specific loading
# ------------------------------------------------------------

for dataset_id in DATASETS:

    path = RAW_FILE_MAP[dataset_id]

    if path is None:
        raise FileNotFoundError(
            f"Raw file not found for dataset: {dataset_id}"
        )

    delimiter = DEFAULT_DELIMITERS.get(
        dataset_id,
        ","
    )

    # --------------------------------------------------------
    # Adult Income — headerless
    # --------------------------------------------------------

    if dataset_id == "adult_income":

        df = pd.read_csv(
            path,
            sep=delimiter,
            header=None,
            names=ADULT_INCOME_COLUMNS,
            skipinitialspace=True,
            na_values=[
                "?",
                " ?",
                "",
                "NA",
                "N/A",
                "null",
                "NULL"
            ],
            keep_default_na=True
        )

        loading_method = (
            "HEADERLESS_LOAD_WITH_CANONICAL_SCHEMA"
        )

        expected_columns = (
            ADULT_INCOME_COLUMNS
        )

    # --------------------------------------------------------
    # Bank Marketing — headered
    # --------------------------------------------------------

    elif dataset_id == "bank_marketing":

        df = pd.read_csv(
            path,
            sep=delimiter,
            header=0,
            skipinitialspace=True,
            na_values=[
                "",
                "NA",
                "N/A",
                "null",
                "NULL"
            ],
            keep_default_na=True
        )

        loading_method = (
            "HEADERED_LOAD"
        )

        expected_columns = None

    # --------------------------------------------------------
    # Diabetes 130-US Hospitals — headered
    # --------------------------------------------------------

    elif dataset_id == "diabetes_130us":

        df = pd.read_csv(
            path,
            sep=delimiter,
            header=0,
            skipinitialspace=True,
            na_values=[
                "?",
                "",
                "NA",
                "N/A",
                "null",
                "NULL"
            ],
            keep_default_na=True
        )

        loading_method = (
            "HEADERED_LOAD"
        )

        expected_columns = None

    # --------------------------------------------------------
    # Unsupported dataset
    # --------------------------------------------------------

    else:

        raise ValueError(
            f"No loading specification defined for "
            f"dataset: {dataset_id}"
        )


    # --------------------------------------------------------
    # Basic cleanup
    # --------------------------------------------------------

    df.columns = [
        str(column).strip()
        for column in df.columns
    ]

    # Remove completely empty trailing rows
    df = df.dropna(
        axis=0,
        how="all"
    ).reset_index(
        drop=True
    )


    # --------------------------------------------------------
    # Adult Income structural verification
    # --------------------------------------------------------

    if dataset_id == "adult_income":

        if list(df.columns) != ADULT_INCOME_COLUMNS:

            raise AssertionError(
                "Adult Income canonical schema was not "
                "assigned correctly."
            )

        if df.shape[1] != 15:

            raise AssertionError(
                f"Adult Income must contain 15 columns. "
                f"Observed: {df.shape[1]}"
            )

        # Critical preservation test:
        # first raw observation must remain in the dataset.
        if df.iloc[0]["age"] != 39:

            raise AssertionError(
                "Adult Income first observation was not "
                "preserved. The raw file may have been "
                "loaded with header=0 instead of header=None."
            )


    # --------------------------------------------------------
    # Store dataset
    # --------------------------------------------------------

    RAW_DATASETS[
        dataset_id
    ] = df


    # --------------------------------------------------------
    # Loading report
    # --------------------------------------------------------

    DATASET_LOAD_ROWS.append(
        {
            "dataset_id":
                dataset_id,

            "file_name":
                path.name,

            "loading_method":
                loading_method,

            "delimiter":
                delimiter,

            "header_mode":
                "header=None"
                if dataset_id in HEADERLESS_DATASETS
                else "header=0",

            "rows":
                int(df.shape[0]),

            "columns":
                int(df.shape[1]),

            "first_column":
                str(df.columns[0]),

            "target_column":
                FEATURE_TARGET_REGISTRY[
                    dataset_id
                ]["target"]
        }
    )


DATASET_LOAD_DF = pd.DataFrame(
    DATASET_LOAD_ROWS
)

display(
    DATASET_LOAD_DF
)


# ------------------------------------------------------------
# Final loading validation
# ------------------------------------------------------------

assert set(
    RAW_DATASETS.keys()
) == set(
    DATASETS
)

for dataset_id, df in RAW_DATASETS.items():

    assert df.shape[0] > 0, (
        f"{dataset_id}: dataset contains no rows."
    )

    assert df.shape[1] > 0, (
        f"{dataset_id}: dataset contains no columns."
    )


print()
print("Dataset loading: PASSED")
print(
    "All raw datasets were loaded using their "
    "canonical structural specifications."
)

,dataset_id,file_name,loading_method,delimiter,header_mode,rows,columns,first_column,target_column
0,adult_income,adult_income.csv,HEADERLESS_LOAD_WITH_CANONICAL_SCHEMA,",",header=None,32561,15,age,income
1,bank_marketing,bank_marketing.csv,HEADERED_LOAD,;,header=0,45211,17,age,y
2,diabetes_130us,diabetes_130us.csv,HEADERED_LOAD,",",header=0,101766,48,race,readmitted



Dataset loading: PASSED
All raw datasets were loaded using their canonical structural specifications.


In [39]:
# ============================================================
# CELL 01.6.1 — DIABETES RAW SCHEMA DIAGNOSTIC
# ============================================================

diabetes_df = RAW_DATASETS["diabetes_130us"]

print("=" * 100)
print("DIABETES 130-US RAW SCHEMA DIAGNOSTIC")
print("=" * 100)

print(f"Rows    : {len(diabetes_df):,}")
print(f"Columns : {len(diabetes_df.columns)}")

print("\nColumns:")
for i, column in enumerate(
    diabetes_df.columns,
    start=1
):
    print(f"{i:02d}. {column}")

DIABETES 130-US RAW SCHEMA DIAGNOSTIC
Rows    : 101,766
Columns : 48

Columns:
01. race
02. gender
03. age
04. weight
05. admission_type_id
06. discharge_disposition_id
07. admission_source_id
08. time_in_hospital
09. payer_code
10. medical_specialty
11. num_lab_procedures
12. num_procedures
13. num_medications
14. number_outpatient
15. number_emergency
16. number_inpatient
17. diag_1
18. diag_2
19. diag_3
20. number_diagnoses
21. max_glu_serum
22. A1Cresult
23. metformin
24. repaglinide
25. nateglinide
26. chlorpropamide
27. glimepiride
28. acetohexamide
29. glipizide
30. glyburide
31. tolbutamide
32. pioglitazone
33. rosiglitazone
34. acarbose
35. miglitol
36. troglitazone
37. tolazamide
38. examide
39. citoglipton
40. insulin
41. glyburide-metformin
42. glipizide-metformin
43. glimepiride-pioglitazone
44. metformin-rosiglitazone
45. metformin-pioglitazone
46. change
47. diabetesMed
48. readmitted


In [40]:
# ============================================================
# CELL 01.6.2 — DIABETES RAW SCHEMA DIAGNOSTIC
# ============================================================

from pathlib import Path
import yaml
import json
import pandas as pd

# ------------------------------------------------------------
# 1. Re-establish project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

CONFIG_DIR = PROJECT_ROOT / "config"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

DATASET_REGISTRY_PATH = (
    CONFIG_DIR / "dataset_registry.yaml"
)

# ------------------------------------------------------------
# 2. Load dataset registry directly from Drive
# ------------------------------------------------------------

if not DATASET_REGISTRY_PATH.exists():

    raise FileNotFoundError(
        f"Dataset registry not found:\n"
        f"{DATASET_REGISTRY_PATH}"
    )

with open(
    DATASET_REGISTRY_PATH,
    "r",
    encoding="utf-8"
) as file:

    DATASET_REGISTRY = yaml.safe_load(file)


# ------------------------------------------------------------
# 3. Resolve dataset registry
# ------------------------------------------------------------

if "datasets" in DATASET_REGISTRY:

    DATASETS_CONFIG = (
        DATASET_REGISTRY["datasets"]
    )

elif "dataset_registry" in DATASET_REGISTRY:

    DATASETS_CONFIG = (
        DATASET_REGISTRY["dataset_registry"]
    )

else:

    DATASETS_CONFIG = DATASET_REGISTRY


# ------------------------------------------------------------
# 4. Verify diabetes configuration
# ------------------------------------------------------------

if "diabetes_130us" not in DATASETS_CONFIG:

    raise KeyError(
        "diabetes_130us is not present in "
        "dataset_registry.yaml"
    )

diabetes_config = DATASETS_CONFIG[
    "diabetes_130us"
]


# ------------------------------------------------------------
# 5. Locate the raw Diabetes file
# ------------------------------------------------------------

configured_file = (
    diabetes_config.get("raw_file")
)

if configured_file is None:

    raise KeyError(
        "diabetes_130us does not contain "
        "'raw_file' in dataset_registry.yaml"
    )

diabetes_path = (
    RAW_DIR / str(configured_file)
)

if not diabetes_path.exists():

    raise FileNotFoundError(
        f"Diabetes raw file not found:\n"
        f"{diabetes_path}"
    )


# ------------------------------------------------------------
# 6. Detect delimiter
# ------------------------------------------------------------

with open(
    diabetes_path,
    "r",
    encoding="utf-8-sig",
    errors="replace"
) as file:

    first_line = file.readline()

delimiter_candidates = {
    ",": first_line.count(","),
    ";": first_line.count(";"),
    "\t": first_line.count("\t"),
    "|": first_line.count("|")
}

delimiter = max(
    delimiter_candidates,
    key=delimiter_candidates.get
)


# ------------------------------------------------------------
# 7. Load raw dataset
# ------------------------------------------------------------

diabetes_df = pd.read_csv(
    diabetes_path,
    sep=delimiter,
    engine="python"
)


# ------------------------------------------------------------
# 8. Display diagnostic information
# ------------------------------------------------------------

print("=" * 100)
print("DIABETES 130-US RAW SCHEMA DIAGNOSTIC")
print("=" * 100)

print(
    f"File      : {diabetes_path}"
)

print(
    f"Delimiter : {repr(delimiter)}"
)

print(
    f"Rows      : {len(diabetes_df):,}"
)

print(
    f"Columns   : {len(diabetes_df.columns)}"
)

print("\nObserved columns:")
print("-" * 100)

for i, column in enumerate(
    diabetes_df.columns,
    start=1
):

    print(
        f"{i:02d}. {column}"
    )


# ------------------------------------------------------------
# 9. Display registry information
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("DIABETES DATASET REGISTRY CONFIGURATION")
print("=" * 100)

print(
    json.dumps(
        diabetes_config,
        indent=4,
        default=str
    )
)

DIABETES 130-US RAW SCHEMA DIAGNOSTIC
File      : /content/drive/MyDrive/AIR_LLM_Research/data/raw/diabetes_130us.csv
Delimiter : ','
Rows      : 101,766
Columns   : 48

Observed columns:
----------------------------------------------------------------------------------------------------
01. race
02. gender
03. age
04. weight
05. admission_type_id
06. discharge_disposition_id
07. admission_source_id
08. time_in_hospital
09. payer_code
10. medical_specialty
11. num_lab_procedures
12. num_procedures
13. num_medications
14. number_outpatient
15. number_emergency
16. number_inpatient
17. diag_1
18. diag_2
19. diag_3
20. number_diagnoses
21. max_glu_serum
22. A1Cresult
23. metformin
24. repaglinide
25. nateglinide
26. chlorpropamide
27. glimepiride
28. acetohexamide
29. glipizide
30. glyburide
31. tolbutamide
32. pioglitazone
33. rosiglitazone
34. acarbose
35. miglitol
36. troglitazone
37. tolazamide
38. examide
39. citoglipton
40. insulin
41. glyburide-metformin
42. glipizide-metformin
43.

In [41]:
# ============================================================
# CELL 01.6.3 — DIABETES SCHEMA SUMMARY
# ============================================================

print("=" * 100)
print("DIABETES SCHEMA SUMMARY")
print("=" * 100)

print(
    f"Observed column count : {len(diabetes_df.columns)}"
)

print(
    f"Registry configuration : "
    f"{diabetes_config}"
)

DIABETES SCHEMA SUMMARY
Observed column count : 48
Registry configuration : {'dataset_name': 'Diabetes 130-US Hospitals', 'raw_file': 'diabetes_130us.csv', 'processed_file': 'diabetes_130us_processed.csv', 'target': 'readmitted', 'task': 'classification', 'domain': 'healthcare', 'expected_mixed_type': True, 'raw_path': '/content/drive/MyDrive/AIR_LLM_Research/01_Raw_Data/diabetes_130us.csv', 'processed_path': '/content/drive/MyDrive/AIR_LLM_Research/02_Preprocessed_Data/diabetes_130us_processed.csv'}


In [42]:
# ============================================================
# CELL 01.7 — HEADER / SCHEMA RECONCILIATION
# ============================================================
#
# Purpose:
#   Reconcile loaded datasets against the canonical schema.
#
#   Headerless datasets receive their canonical schema.
#   Headered datasets retain their original headers.
#
#   No observations are discarded during reconciliation.
# ============================================================

SCHEMA_RECONCILIATION_ROWS = []

DATASET_SCHEMAS = {}

STANDARDIZED_DATASETS = {}


# ------------------------------------------------------------
# Canonical schema definitions
# ------------------------------------------------------------

CANONICAL_SCHEMAS = {

    "adult_income":
        ADULT_INCOME_COLUMNS
}


# ------------------------------------------------------------
# Reconcile each dataset
# ------------------------------------------------------------

for dataset_id in DATASETS:

    df = RAW_DATASETS[
        dataset_id
    ].copy()

    original_row_count = (
        df.shape[0]
    )

    original_column_count = (
        df.shape[1]
    )

    observed_columns = list(
        df.columns
    )


    # --------------------------------------------------------
    # Headerless / canonical-schema dataset
    # --------------------------------------------------------

    if dataset_id in CANONICAL_SCHEMAS:

        expected_columns = list(
            CANONICAL_SCHEMAS[
                dataset_id
            ]
        )

        expected_column_count = len(
            expected_columns
        )

        if original_column_count != expected_column_count:

            raise ValueError(
                f"{dataset_id}: canonical schema mismatch.\n"
                f"Observed columns : "
                f"{original_column_count}\n"
                f"Expected columns : "
                f"{expected_column_count}\n"
                f"Expected schema  : "
                f"{expected_columns}"
            )

        # Assign canonical names without changing rows.
        df.columns = expected_columns

        final_columns = list(
            df.columns
        )

        schema_status = (
            "RECONSTRUCTED_HEADERLESS_SCHEMA"
        )

        reconstruction_method = (
            "CANONICAL_SCHEMA_ASSIGNED"
        )

        missing_expected_columns = []

        unexpected_columns = []


    # --------------------------------------------------------
    # Headered datasets
    # --------------------------------------------------------

    else:

        expected_columns = None

        expected_column_count = None

        final_columns = list(
            df.columns
        )

        schema_status = (
            "OBSERVED_SCHEMA_ACCEPTED"
        )

        reconstruction_method = (
            "EXISTING_HEADER_PRESERVED"
        )

        missing_expected_columns = []

        unexpected_columns = []


    # --------------------------------------------------------
    # Verify row preservation
    # --------------------------------------------------------

    if df.shape[0] != original_row_count:

        raise AssertionError(
            f"{dataset_id}: row count changed during "
            f"schema reconciliation."
        )


    # --------------------------------------------------------
    # Verify column uniqueness
    # --------------------------------------------------------

    if df.columns.duplicated().any():

        duplicated_columns = (
            df.columns[
                df.columns.duplicated()
            ].tolist()
        )

        raise ValueError(
            f"{dataset_id}: duplicate column names detected: "
            f"{duplicated_columns}"
        )


    # --------------------------------------------------------
    # Store reconciled dataset
    # --------------------------------------------------------

    STANDARDIZED_DATASETS[
        dataset_id
    ] = df


    # --------------------------------------------------------
    # Store schema metadata
    # --------------------------------------------------------

    DATASET_SCHEMAS[
        dataset_id
    ] = {

        "columns":
            final_columns,

        "column_count":
            len(final_columns),

        "expected_columns":
            expected_columns,

        "expected_column_count":
            expected_column_count,

        "schema_status":
            schema_status,

        "reconstruction_method":
            reconstruction_method,

        "row_count":
            int(df.shape[0])
    }


    # --------------------------------------------------------
    # Reconciliation report
    # --------------------------------------------------------

    SCHEMA_RECONCILIATION_ROWS.append(
        {

            "dataset_id":
                dataset_id,

            "expected_count":
                expected_column_count,

            "observed_count":
                original_column_count,

            "final_count":
                len(final_columns),

            "missing_expected_columns":
                missing_expected_columns,

            "unexpected_columns":
                unexpected_columns,

            "schema_status":
                schema_status,

            "reconstruction_method":
                reconstruction_method,

            "row_count":
                int(df.shape[0])
        }
    )


SCHEMA_RECONCILIATION_DF = pd.DataFrame(
    SCHEMA_RECONCILIATION_ROWS
)

display(
    SCHEMA_RECONCILIATION_DF
)


# ------------------------------------------------------------
# Critical Adult Income validation
# ------------------------------------------------------------

adult_df = STANDARDIZED_DATASETS[
    "adult_income"
]

assert list(
    adult_df.columns
) == ADULT_INCOME_COLUMNS, (
    "Adult Income canonical columns are incorrect."
)

assert (
    adult_df.shape[1] == 15
), (
    "Adult Income must contain exactly 15 columns."
)

assert (
    adult_df.iloc[0]["age"] == 39
), (
    "Adult Income first observation was not preserved. "
    "Expected age=39 in the first raw observation."
)

assert (
    adult_df.iloc[0]["workclass"]
    == "State-gov"
), (
    "Adult Income first observation was not preserved. "
    "Expected workclass='State-gov'."
)

assert (
    adult_df.iloc[0]["fnlwgt"]
    == 77516
), (
    "Adult Income first observation was not preserved. "
    "Expected fnlwgt=77516."
)


# ------------------------------------------------------------
# Dataset-level validation
# ------------------------------------------------------------

for dataset_id in DATASETS:

    df = STANDARDIZED_DATASETS[
        dataset_id
    ]

    assert df.shape[0] > 0, (
        f"{dataset_id}: no observations remain."
    )

    assert df.shape[1] > 0, (
        f"{dataset_id}: no features remain."
    )


print()
print("Schema reconciliation: PASSED")
print(
    "Canonical headerless schemas were reconstructed "
    "without loss of observations."
)
print(
    "Adult Income first observation preservation: PASSED"
)

,dataset_id,expected_count,observed_count,final_count,missing_expected_columns,unexpected_columns,schema_status,reconstruction_method,row_count
0,adult_income,15.0,15,15,[],[],RECONSTRUCTED_HEADERLESS_SCHEMA,CANONICAL_SCHEMA_ASSIGNED,32561
1,bank_marketing,NaN,17,17,[],[],OBSERVED_SCHEMA_ACCEPTED,EXISTING_HEADER_PRESERVED,45211
2,diabetes_130us,NaN,48,48,[],[],OBSERVED_SCHEMA_ACCEPTED,EXISTING_HEADER_PRESERVED,101766



Schema reconciliation: PASSED
Canonical headerless schemas were reconstructed without loss of observations.
Adult Income first observation preservation: PASSED


In [47]:
# ============================================================
# CELL 01.8 — COLUMN NAME STANDARDIZATION
# ============================================================
#
# Purpose:
#   Standardize column names consistently across all datasets
#   while preserving complete traceability to the raw schema.
#
# Important:
#   No rows, values, or features are removed in this cell.
# ============================================================

import re

COLUMN_STANDARDIZATION_ROWS = []
COLUMN_NAME_MAP = {}
STANDARDIZED_DATASETS = {}


def standardize_column_name(column):

    column = str(column).strip()

    column = (
        column
        .replace("?", "")
        .replace(".", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )

    column = re.sub(
        r"[^A-Za-z0-9_]+",
        "_",
        column
    )

    column = re.sub(
        r"_+",
        "_",
        column
    )

    column = column.strip("_").lower()

    if not column:
        column = "unnamed_feature"

    return column


for dataset_id in DATASETS:

    df = RAW_DATASETS[dataset_id].copy()

    original_columns = list(df.columns)

    standardized_columns = [
        standardize_column_name(column)
        for column in original_columns
    ]

    # --------------------------------------------------------
    # Duplicate protection
    # --------------------------------------------------------

    if len(standardized_columns) != len(
        set(standardized_columns)
    ):

        counts = (
            pd.Series(standardized_columns)
            .value_counts()
        )

        duplicates = counts[
            counts > 1
        ].index.tolist()

        raise ValueError(
            f"{dataset_id}: duplicate column names after "
            f"standardization: {duplicates}"
        )

    # --------------------------------------------------------
    # Apply standardized names
    # --------------------------------------------------------

    df.columns = standardized_columns

    STANDARDIZED_DATASETS[dataset_id] = df

    COLUMN_NAME_MAP[dataset_id] = dict(
        zip(
            original_columns,
            standardized_columns
        )
    )

    # --------------------------------------------------------
    # Traceability log
    # --------------------------------------------------------

    for original, standardized in zip(
        original_columns,
        standardized_columns
    ):

        COLUMN_STANDARDIZATION_ROWS.append({

            "dataset_id":
                dataset_id,

            "original_column":
                str(original),

            "standardized_column":
                standardized,

            "changed":
                str(original) != standardized
        })


COLUMN_STANDARDIZATION_DF = pd.DataFrame(
    COLUMN_STANDARDIZATION_ROWS
)

display(
    COLUMN_STANDARDIZATION_DF
)


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

for dataset_id, df in STANDARDIZED_DATASETS.items():

    if df.columns.duplicated().any():

        duplicates = (
            df.columns[
                df.columns.duplicated()
            ].tolist()
        )

        raise ValueError(
            f"{dataset_id}: duplicate standardized "
            f"columns detected: {duplicates}"
        )

    if len(df.columns) != len(
        RAW_DATASETS[dataset_id].columns
    ):

        raise ValueError(
            f"{dataset_id}: column count changed "
            "during standardization."
        )


print("=" * 100)
print("COLUMN NAME STANDARDIZATION: PASSED")
print("=" * 100)
print(
    f"Datasets processed : {len(STANDARDIZED_DATASETS)}"
)
print(
    f"Columns mapped     : "
    f"{len(COLUMN_STANDARDIZATION_DF)}"
)
print(
    "No rows or values were modified."
)

,dataset_id,original_column,standardized_column,changed
0,adult_income,age,age,False
1,adult_income,workclass,workclass,False
2,adult_income,fnlwgt,fnlwgt,False
3,adult_income,education,education,False
4,adult_income,education_num,education_num,False
...,...,...,...,...
75,diabetes_130us,metformin-rosiglitazone,metformin_rosiglitazone,True
76,diabetes_130us,metformin-pioglitazone,metformin_pioglitazone,True
77,diabetes_130us,change,change,False
78,diabetes_130us,diabetesMed,diabetesmed,True


COLUMN NAME STANDARDIZATION: PASSED
Datasets processed : 3
Columns mapped     : 80
No rows or values were modified.


In [48]:
# ============================================================
# CELL 01.9 — TARGET VALIDATION
# ============================================================

TARGET_VALIDATION_ROWS = []

for dataset_id in DATASETS:

    df = STANDARDIZED_DATASETS[dataset_id]

    configured_target = (
        FEATURE_TARGET_REGISTRY[
            dataset_id
        ]["target"]
    )

    target_exists = (
        configured_target in df.columns
    )

    if target_exists:

        target_series = df[
            configured_target
        ]

        target_dtype = str(
            target_series.dtype
        )

        target_missing = int(
            target_series.isna().sum()
        )

        target_unique = int(
            target_series.nunique(
                dropna=False
            )
        )

        target_values = sorted(
            target_series
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        target_missing_rate = float(
            target_series.isna().mean()
        )

    else:

        target_dtype = None
        target_missing = None
        target_unique = None
        target_values = []
        target_missing_rate = None

    TARGET_VALIDATION_ROWS.append({

        "dataset_id":
            dataset_id,

        "target":
            configured_target,

        "target_exists":
            target_exists,

        "target_dtype":
            target_dtype,

        "target_missing_count":
            target_missing,

        "target_missing_rate":
            target_missing_rate,

        "target_unique_values":
            target_unique,

        "target_values":
            target_values
    })


TARGET_VALIDATION_DF = pd.DataFrame(
    TARGET_VALIDATION_ROWS
)

display(
    TARGET_VALIDATION_DF
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

if not TARGET_VALIDATION_DF[
    "target_exists"
].all():

    invalid = (
        TARGET_VALIDATION_DF.loc[
            ~TARGET_VALIDATION_DF[
                "target_exists"
            ],
            "dataset_id"
        ].tolist()
    )

    raise ValueError(
        f"Target missing for datasets: {invalid}"
    )


if not TARGET_VALIDATION_DF[
    "target_missing_count"
].eq(0).all():

    invalid = (
        TARGET_VALIDATION_DF.loc[
            TARGET_VALIDATION_DF[
                "target_missing_count"
            ] > 0,
            [
                "dataset_id",
                "target",
                "target_missing_count"
            ]
        ]
        .to_dict("records")
    )

    raise ValueError(
        "Target contains missing values:\n"
        f"{invalid}"
    )


print("=" * 100)
print("TARGET VALIDATION: PASSED")
print("=" * 100)

,dataset_id,target,target_exists,target_dtype,target_missing_count,target_missing_rate,target_unique_values,target_values
0,adult_income,income,True,object,0,0.0,2,"[<=50K, >50K]"
1,bank_marketing,y,True,object,0,0.0,2,"[no, yes]"
2,diabetes_130us,readmitted,True,object,0,0.0,3,"[<30, >30, NO]"


TARGET VALIDATION: PASSED


In [49]:
# ============================================================
# CELL 01.10 — FEATURE TYPE DETECTION
# ============================================================

FEATURE_TYPE_ROWS = []
FEATURE_REGISTRY = {}


for dataset_id in DATASETS:

    df = STANDARDIZED_DATASETS[dataset_id]

    target = (
        FEATURE_TARGET_REGISTRY[
            dataset_id
        ]["target"]
    )

    all_features = [
        column
        for column in df.columns
        if column != target
    ]

    numeric_features = [
        column
        for column in all_features
        if pd.api.types.is_numeric_dtype(
            df[column]
        )
    ]

    categorical_features = [
        column
        for column in all_features
        if (
            pd.api.types.is_object_dtype(df[column])
            or pd.api.types.is_categorical_dtype(df[column])
            or pd.api.types.is_bool_dtype(df[column])
        )
    ]

    unsupported_features = [
        column
        for column in all_features
        if (
            column not in numeric_features
            and column not in categorical_features
        )
    ]

    # --------------------------------------------------------
    # Registry
    # --------------------------------------------------------

    FEATURE_REGISTRY[dataset_id] = {

        "target":
            target,

        "numeric_features":
            numeric_features,

        "categorical_features":
            categorical_features,

        "unsupported_features":
            unsupported_features,

        "all_features":
            all_features,

        "feature_count":
            len(all_features),

        "numeric_count":
            len(numeric_features),

        "categorical_count":
            len(categorical_features)
    }

    # --------------------------------------------------------
    # Feature-level profile
    # --------------------------------------------------------

    for column in df.columns:

        if column == target:

            feature_type = "target"

        elif column in numeric_features:

            feature_type = "numerical"

        elif column in categorical_features:

            feature_type = "categorical"

        else:

            feature_type = "unsupported"

        series = df[column]

        FEATURE_TYPE_ROWS.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "dtype":
                str(series.dtype),

            "feature_type":
                feature_type,

            "is_target":
                column == target,

            "n_unique":
                int(
                    series.nunique(
                        dropna=False
                    )
                ),

            "missing_count":
                int(
                    series.isna().sum()
                ),

            "missing_rate":
                float(
                    series.isna().mean()
                )
        })


FEATURE_TYPE_DF = pd.DataFrame(
    FEATURE_TYPE_ROWS
)

display(
    FEATURE_TYPE_DF
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

for dataset_id in DATASETS:

    registry = FEATURE_REGISTRY[
        dataset_id
    ]

    if registry[
        "unsupported_features"
    ]:

        raise TypeError(
            f"{dataset_id}: unsupported feature types: "
            f"{registry['unsupported_features']}"
        )

    if (
        len(registry["numeric_features"])
        + len(registry["categorical_features"])
        != len(registry["all_features"])
    ):

        raise AssertionError(
            f"{dataset_id}: feature-type partition incomplete."
        )


print("=" * 100)
print("FEATURE-TYPE DETECTION: PASSED")
print("=" * 100)

,dataset_id,feature,dtype,feature_type,is_target,n_unique,missing_count,missing_rate
0,adult_income,age,int64,numerical,False,73,0,0.000000
1,adult_income,workclass,object,categorical,False,9,1836,0.056386
2,adult_income,fnlwgt,int64,numerical,False,21648,0,0.000000
3,adult_income,education,object,categorical,False,16,0,0.000000
4,adult_income,education_num,int64,numerical,False,16,0,0.000000
...,...,...,...,...,...,...,...,...
75,diabetes_130us,metformin_rosiglitazone,object,categorical,False,2,0,0.000000
76,diabetes_130us,metformin_pioglitazone,object,categorical,False,2,0,0.000000
77,diabetes_130us,change,object,categorical,False,2,0,0.000000
78,diabetes_130us,diabetesmed,object,categorical,False,2,0,0.000000


FEATURE-TYPE DETECTION: PASSED


In [50]:
# ============================================================
# CELL 01.11 — IDENTIFIER / CONSTANT FEATURE DETECTION
# ============================================================

IDENTIFIER_ROWS = []
IDENTIFIER_REGISTRY = {}


def detect_identifier_like(
    series,
    column_name
):

    non_missing = series.dropna()

    if len(non_missing) == 0:
        return False, "ALL_VALUES_MISSING"

    unique_ratio = (
        non_missing.nunique()
        / len(non_missing)
    )

    name_tokens = set(
        re.split(
            r"[_\s]+",
            column_name.lower()
        )
    )

    identifier_tokens = {
        "id",
        "identifier",
        "encounter",
        "patient",
        "record",
        "index"
    }

    name_based = bool(
        name_tokens & identifier_tokens
    )

    high_cardinality = (
        unique_ratio >= 0.98
        and len(non_missing) >= 20
    )

    if name_based:
        return True, "NAME_BASED"

    if high_cardinality:
        return True, "HIGH_CARDINALITY"

    return False, None


for dataset_id in DATASETS:

    df = STANDARDIZED_DATASETS[dataset_id]

    target = FEATURE_REGISTRY[
        dataset_id
    ]["target"]

    dataset_identifiers = []
    dataset_constants = []

    for column in df.columns:

        series = df[column]

        non_missing = series.dropna()

        unique_count = int(
            series.nunique(
                dropna=False
            )
        )

        non_missing_unique = int(
            non_missing.nunique()
        )

        unique_ratio = (
            non_missing_unique
            / len(non_missing)
            if len(non_missing) > 0
            else 0.0
        )

        constant = (
            unique_count <= 1
        )

        identifier_like, identifier_reason = (
            detect_identifier_like(
                series,
                column
            )
        )

        # Target is never flagged as an
        # identifier for removal.
        if column == target:
            identifier_like = False
            identifier_reason = None

        if constant:
            dataset_constants.append(column)

        if identifier_like:
            dataset_identifiers.append(column)

        IDENTIFIER_ROWS.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "is_target":
                column == target,

            "unique_count":
                unique_count,

            "non_missing_unique_count":
                non_missing_unique,

            "unique_ratio":
                round(
                    unique_ratio,
                    6
                ),

            "constant_feature":
                constant,

            "identifier_like":
                identifier_like,

            "identifier_reason":
                identifier_reason
        })

    IDENTIFIER_REGISTRY[
        dataset_id
    ] = {

        "identifier_like_features":
            dataset_identifiers,

        "constant_features":
            dataset_constants
    }


IDENTIFIER_DF = pd.DataFrame(
    IDENTIFIER_ROWS
)

display(
    IDENTIFIER_DF
)


print("=" * 100)
print("IDENTIFIER / CONSTANT FEATURE DETECTION: PASSED")
print("=" * 100)

print(
    "No features were removed. "
    "All detections are retained as metadata for downstream notebooks."
)

,dataset_id,feature,is_target,unique_count,non_missing_unique_count,unique_ratio,constant_feature,identifier_like,identifier_reason
0,adult_income,age,False,73,73,0.002242,False,False,None
1,adult_income,workclass,False,9,8,0.000260,False,False,None
2,adult_income,fnlwgt,False,21648,21648,0.664844,False,False,None
3,adult_income,education,False,16,16,0.000491,False,False,None
4,adult_income,education_num,False,16,16,0.000491,False,False,None
...,...,...,...,...,...,...,...,...,...
75,diabetes_130us,metformin_rosiglitazone,False,2,2,0.000020,False,False,None
76,diabetes_130us,metformin_pioglitazone,False,2,2,0.000020,False,False,None
77,diabetes_130us,change,False,2,2,0.000020,False,False,None
78,diabetes_130us,diabetesmed,False,2,2,0.000020,False,False,None


IDENTIFIER / CONSTANT FEATURE DETECTION: PASSED
No features were removed. All detections are retained as metadata for downstream notebooks.


In [51]:
# ============================================================
# CELL 01.12 — DATASET STRUCTURAL PROFILING
# ============================================================
#
# Purpose:
#   Create the authoritative structural profile of every raw
#   dataset after schema reconstruction and column standardization.
#
# Important:
#   No imputation, encoding, scaling, outlier treatment, or
#   missingness generation is performed here.
# ============================================================

STRUCTURAL_ROWS = []


for dataset_id in DATASETS:

    df = STANDARDIZED_DATASETS[dataset_id]

    target = FEATURE_REGISTRY[
        dataset_id
    ]["target"]

    numeric_features = FEATURE_REGISTRY[
        dataset_id
    ]["numeric_features"]

    categorical_features = FEATURE_REGISTRY[
        dataset_id
    ]["categorical_features"]

    total_cells = int(
        df.shape[0] * df.shape[1]
    )

    missing_cells = int(
        df.isna()
        .sum()
        .sum()
    )

    duplicate_rows = int(
        df.duplicated()
        .sum()
    )

    constant_features = (
        IDENTIFIER_REGISTRY[
            dataset_id
        ]["constant_features"]
    )

    identifier_features = (
        IDENTIFIER_REGISTRY[
            dataset_id
        ]["identifier_like_features"]
    )

    target_missing = int(
        df[target].isna().sum()
    )

    target_unique = int(
        df[target].nunique(
            dropna=False
        )
    )

    STRUCTURAL_ROWS.append({

        "dataset_id":
            dataset_id,

        "dataset_name":
            DATASET_REGISTRY[
                dataset_id
            ]["dataset_name"],

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "numeric_features":
            len(numeric_features),

        "categorical_features":
            len(categorical_features),

        "target":
            target,

        "target_dtype":
            str(df[target].dtype),

        "target_unique_values":
            target_unique,

        "target_missing_count":
            target_missing,

        "missing_cells":
            missing_cells,

        "missing_cell_rate":
            (
                missing_cells / total_cells
                if total_cells > 0
                else 0.0
            ),

        "complete_rows":
            int(
                df.notna()
                .all(axis=1)
                .sum()
            ),

        "duplicate_rows":
            duplicate_rows,

        "duplicate_row_rate":
            (
                duplicate_rows / len(df)
                if len(df) > 0
                else 0.0
            ),

        "constant_features":
            len(constant_features),

        "identifier_like_features":
            len(identifier_features),

        "memory_mb":
            round(
                df.memory_usage(
                    deep=True
                ).sum()
                / 1024**2,
                3
            )
    })


STRUCTURAL_PROFILE_DF = pd.DataFrame(
    STRUCTURAL_ROWS
)

display(
    STRUCTURAL_PROFILE_DF
)


# ------------------------------------------------------------
# Structural integrity validation
# ------------------------------------------------------------

for dataset_id in DATASETS:

    df = STANDARDIZED_DATASETS[
        dataset_id
    ]

    if df.empty:

        raise ValueError(
            f"{dataset_id}: dataset is empty."
        )

    if df.shape[1] == 0:

        raise ValueError(
            f"{dataset_id}: dataset contains no columns."
        )

    if df.columns.duplicated().any():

        raise ValueError(
            f"{dataset_id}: duplicate columns detected."
        )

    target = FEATURE_REGISTRY[
        dataset_id
    ]["target"]

    if target not in df.columns:

        raise ValueError(
            f"{dataset_id}: target '{target}' "
            "is absent from structural profile."
        )


print("=" * 100)
print("DATASET STRUCTURAL PROFILING: PASSED")
print("=" * 100)

print(
    f"Datasets profiled : {len(STRUCTURAL_PROFILE_DF)}"
)
print(
    f"Total rows        : "
    f"{STRUCTURAL_PROFILE_DF['rows'].sum():,}"
)
print(
    f"Total columns     : "
    f"{STRUCTURAL_PROFILE_DF['columns'].sum():,}"
)

,dataset_id,dataset_name,rows,columns,numeric_features,categorical_features,target,target_dtype,target_unique_values,target_missing_count,missing_cells,missing_cell_rate,complete_rows,duplicate_rows,duplicate_row_rate,constant_features,identifier_like_features,memory_mb
0,adult_income,Adult Income,32561,15,6,8,income,object,2,0,4262,0.008726,30162,24,0.000737,0,0,17.574
1,bank_marketing,Bank Marketing,45211,17,7,9,y,object,2,0,0,0.000000,45211,0,0.000000,0,0,25.749
2,diabetes_130us,Diabetes 130-US Hospitals,101766,48,11,36,readmitted,object,3,0,374017,0.076568,0,0,0.000000,2,3,188.007


DATASET STRUCTURAL PROFILING: PASSED
Datasets profiled : 3
Total rows        : 179,538
Total columns     : 80


In [52]:
# ============================================================
# CELL 01.13 — RAW DATASET QUALITY VALIDATION
# ============================================================
#
# Purpose:
#   Perform non-destructive quality validation of the
#   standardized raw datasets.
#
# Important:
#   This cell ONLY detects and reports quality issues.
#   No rows, values, missing observations, or features are
#   modified or removed.
#
# Downstream treatment is performed in Notebook 02.
# ============================================================

QUALITY_ROWS = []


for dataset_id in DATASETS:

    df = STANDARDIZED_DATASETS[
        dataset_id
    ]

    target = FEATURE_REGISTRY[
        dataset_id
    ]["target"]

    # --------------------------------------------------------
    # Basic structural checks
    # --------------------------------------------------------

    row_count = int(
        df.shape[0]
    )

    column_count = int(
        df.shape[1]
    )

    duplicate_rows = int(
        df.duplicated()
        .sum()
    )

    duplicate_row_rate = (
        duplicate_rows / row_count
        if row_count > 0
        else 0.0
    )

    duplicate_columns = int(
        df.columns.duplicated()
        .sum()
    )

    target_exists = (
        target in df.columns
    )

    # --------------------------------------------------------
    # Missing-value diagnostics
    # --------------------------------------------------------

    missing_cells = int(
        df.isna()
        .sum()
        .sum()
    )

    missing_cell_rate = (
        missing_cells
        / (row_count * column_count)
        if row_count > 0 and column_count > 0
        else 0.0
    )

    missing_features = int(
        df.isna()
        .any(axis=0)
        .sum()
    )

    complete_rows = int(
        df.notna()
        .all(axis=1)
        .sum()
    )

    incomplete_rows = (
        row_count
        - complete_rows
    )

    # --------------------------------------------------------
    # Target validation
    # --------------------------------------------------------

    if target_exists:

        missing_target = int(
            df[target]
            .isna()
            .sum()
        )

        target_unique = int(
            df[target]
            .nunique(
                dropna=False
            )
        )

    else:

        missing_target = None
        target_unique = None

    # --------------------------------------------------------
    # Empty-string diagnostics
    # --------------------------------------------------------
    #
    # Only object/string columns are inspected.
    # This avoids converting the complete dataset to strings,
    # which could obscure genuine numeric values and NaNs.
    # --------------------------------------------------------

    string_columns = df.select_dtypes(
        include=["object", "string"]
    )

    empty_string_cells = 0

    whitespace_only_cells = 0

    if not string_columns.empty:

        empty_string_cells = int(
            string_columns
            .apply(
                lambda column:
                column.astype("string")
                .str.strip()
                .eq("")
                .sum()
            )
            .sum()
        )

        whitespace_only_cells = int(
            string_columns
            .apply(
                lambda column:
                (
                    column.astype("string")
                    .str.len()
                    .fillna(0)
                    .eq(0)
                    &
                    column.notna()
                ).sum()
            )
            .sum()
        )

    # --------------------------------------------------------
    # Infinite-value diagnostics
    # --------------------------------------------------------

    numeric_df = df.select_dtypes(
        include=[np.number]
    )

    infinite_values = 0

    if not numeric_df.empty:

        numeric_array = (
            numeric_df
            .to_numpy(
                dtype=float
            )
        )

        infinite_values = int(
            np.isinf(
                numeric_array
            ).sum()
        )

    # --------------------------------------------------------
    # Fully-null columns
    # --------------------------------------------------------

    fully_null_columns = [
        column
        for column in df.columns
        if df[column].isna().all()
    ]

    # --------------------------------------------------------
    # Fully-null rows
    # --------------------------------------------------------

    fully_null_rows = int(
        df.isna()
        .all(axis=1)
        .sum()
    )

    # --------------------------------------------------------
    # Structural quality status
    # --------------------------------------------------------

    quality_schema_valid = (
        row_count > 0
        and column_count > 0
        and duplicate_columns == 0
        and target_exists
    )

    # --------------------------------------------------------
    # Data quality status
    # --------------------------------------------------------
    #
    # Missing values and duplicate rows are NOT considered
    # validation failures because they are legitimate
    # properties that downstream notebooks may handle.
    #
    # Infinite values and completely null columns are flagged.
    # --------------------------------------------------------

    quality_data_valid = (
        infinite_values == 0
        and len(fully_null_columns) == 0
    )

    overall_quality_valid = (
        quality_schema_valid
        and quality_data_valid
    )

    QUALITY_ROWS.append({

        "dataset_id":
            dataset_id,

        "rows":
            row_count,

        "columns":
            column_count,

        "duplicate_rows":
            duplicate_rows,

        "duplicate_row_rate":
            round(
                duplicate_row_rate,
                6
            ),

        "missing_cells":
            missing_cells,

        "missing_cell_rate":
            round(
                missing_cell_rate,
                6
            ),

        "missing_features":
            missing_features,

        "complete_rows":
            complete_rows,

        "incomplete_rows":
            incomplete_rows,

        "missing_target":
            missing_target,

        "target_unique_values":
            target_unique,

        "empty_string_cells":
            empty_string_cells,

        "whitespace_only_cells":
            whitespace_only_cells,

        "infinite_numeric_values":
            infinite_values,

        "fully_null_columns":
            len(fully_null_columns),

        "fully_null_rows":
            fully_null_rows,

        "duplicate_columns":
            duplicate_columns,

        "target_exists":
            target_exists,

        "quality_schema_valid":
            quality_schema_valid,

        "quality_data_valid":
            quality_data_valid,

        "overall_quality_valid":
            overall_quality_valid
    })


RAW_QUALITY_DF = pd.DataFrame(
    QUALITY_ROWS
)

display(
    RAW_QUALITY_DF
)


# ============================================================
# VALIDATION
# ============================================================

if not RAW_QUALITY_DF[
    "quality_schema_valid"
].all():

    invalid = (
        RAW_QUALITY_DF.loc[
            ~RAW_QUALITY_DF[
                "quality_schema_valid"
            ],
            "dataset_id"
        ].tolist()
    )

    raise ValueError(
        "Structural quality validation failed for: "
        f"{invalid}"
    )


if not RAW_QUALITY_DF[
    "quality_data_valid"
].all():

    invalid = (
        RAW_QUALITY_DF.loc[
            ~RAW_QUALITY_DF[
                "quality_data_valid"
            ],
            [
                "dataset_id",
                "infinite_numeric_values",
                "fully_null_columns"
            ]
        ]
        .to_dict("records")
    )

    raise ValueError(
        "Raw data quality validation failed:\n"
        f"{invalid}"
    )


# ============================================================
# FINAL REPORT
# ============================================================

print("=" * 100)
print("RAW DATASET QUALITY VALIDATION: PASSED")
print("=" * 100)

print(
    f"Datasets validated : "
    f"{len(RAW_QUALITY_DF)}"
)

print(
    f"Duplicate rows     : "
    f"{int(RAW_QUALITY_DF['duplicate_rows'].sum()):,}"
)

print(
    f"Missing cells      : "
    f"{int(RAW_QUALITY_DF['missing_cells'].sum()):,}"
)

print(
    f"Infinite values    : "
    f"{int(RAW_QUALITY_DF['infinite_numeric_values'].sum()):,}"
)

print()
print(
    "Missing values and duplicate observations were "
    "reported but NOT removed."
)

print(
    "Empty strings were reported but NOT modified."
)

print(
    "No imputation, encoding, scaling, outlier treatment, "
    "or row filtering was performed."
)

,dataset_id,rows,columns,duplicate_rows,duplicate_row_rate,missing_cells,missing_cell_rate,missing_features,complete_rows,incomplete_rows,...,empty_string_cells,whitespace_only_cells,infinite_numeric_values,fully_null_columns,fully_null_rows,duplicate_columns,target_exists,quality_schema_valid,quality_data_valid,overall_quality_valid
0,adult_income,32561,15,24,0.000737,4262,0.008726,3,30162,2399,...,0,0,0,0,0,0,True,True,True,True
1,bank_marketing,45211,17,0,0.000000,0,0.000000,0,45211,0,...,0,0,0,0,0,0,True,True,True,True
2,diabetes_130us,101766,48,0,0.000000,374017,0.076568,9,0,101766,...,0,0,0,0,0,0,True,True,True,True


RAW DATASET QUALITY VALIDATION: PASSED
Datasets validated : 3
Duplicate rows     : 24
Missing cells      : 378,279
Infinite values    : 0

Missing values and duplicate observations were reported but NOT removed.
Empty strings were reported but NOT modified.
No imputation, encoding, scaling, outlier treatment, or row filtering was performed.


In [53]:
# ============================================================
# CELL 01.14 — DATASET PREPARATION SUMMARY
# ============================================================
#
# Purpose:
#   Consolidate all Notebook 01 acquisition, schema, structural,
#   feature-type and quality diagnostics into one authoritative
#   dataset-level hand-off summary.
#
# Important:
#   This cell does NOT modify the datasets.
#   It only summarizes the validated raw-data state.
# ============================================================

PREPARATION_SUMMARY_ROWS = []


for dataset_id in DATASETS:

    # --------------------------------------------------------
    # Core metadata
    # --------------------------------------------------------

    metadata = DATASET_REGISTRY[
        dataset_id
    ]

    feature_info = FEATURE_REGISTRY[
        dataset_id
    ]

    identifier_info = IDENTIFIER_REGISTRY[
        dataset_id
    ]

    # --------------------------------------------------------
    # Structural profile
    # --------------------------------------------------------

    profile_matches = STRUCTURAL_PROFILE_DF.loc[
        STRUCTURAL_PROFILE_DF[
            "dataset_id"
        ] == dataset_id
    ]

    if profile_matches.empty:

        raise ValueError(
            f"{dataset_id}: structural profile not found."
        )

    profile = profile_matches.iloc[0]

    # --------------------------------------------------------
    # Schema information
    # --------------------------------------------------------

    schema_matches = SCHEMA_RECONCILIATION_DF.loc[
        SCHEMA_RECONCILIATION_DF[
            "dataset_id"
        ] == dataset_id
    ]

    if schema_matches.empty:

        raise ValueError(
            f"{dataset_id}: schema reconciliation record not found."
        )

    schema = schema_matches.iloc[0]

    # --------------------------------------------------------
    # Quality information
    # --------------------------------------------------------

    quality_matches = RAW_QUALITY_DF.loc[
        RAW_QUALITY_DF[
            "dataset_id"
        ] == dataset_id
    ]

    if quality_matches.empty:

        raise ValueError(
            f"{dataset_id}: quality validation record not found."
        )

    quality = quality_matches.iloc[0]

    # --------------------------------------------------------
    # Raw file information
    # --------------------------------------------------------

    raw_file = RAW_FILE_MAP[
        dataset_id
    ]

    if raw_file is None:

        raise FileNotFoundError(
            f"{dataset_id}: raw file not available."
        )

    # --------------------------------------------------------
    # Dataset summary
    # --------------------------------------------------------

    PREPARATION_SUMMARY_ROWS.append({

        # ----------------------------------------------------
        # Dataset identity
        # ----------------------------------------------------

        "dataset_id":
            dataset_id,

        "dataset_name":
            metadata.get(
                "dataset_name"
            ),

        "task":
            metadata.get(
                "task"
            ),

        "domain":
            metadata.get(
                "domain"
            ),

        # ----------------------------------------------------
        # Raw source
        # ----------------------------------------------------

        "raw_file":
            raw_file.name,

        "raw_file_format":
            raw_file.suffix.lower(),

        "raw_file_size_bytes":
            int(
                raw_file.stat().st_size
            ),

        # ----------------------------------------------------
        # Schema
        # ----------------------------------------------------

        "schema_status":
            schema["schema_status"],

        "reconstruction_method":
            schema.get(
                "reconstruction_method",
                None
            ),

        "rows":
            int(
                profile["rows"]
            ),

        "columns":
            int(
                profile["columns"]
            ),

        # ----------------------------------------------------
        # Target
        # ----------------------------------------------------

        "target":
            feature_info["target"],

        "target_dtype":
            profile["target_dtype"],

        "target_unique_values":
            int(
                profile["target_unique_values"]
            ),

        "target_missing_count":
            int(
                profile["target_missing_count"]
            ),

        # ----------------------------------------------------
        # Feature types
        # ----------------------------------------------------

        "numeric_features":
            len(
                feature_info[
                    "numeric_features"
                ]
            ),

        "categorical_features":
            len(
                feature_info[
                    "categorical_features"
                ]
            ),

        "total_features":
            len(
                feature_info[
                    "all_features"
                ]
            ),

        # ----------------------------------------------------
        # Missingness
        # ----------------------------------------------------

        "missing_cells":
            int(
                profile["missing_cells"]
            ),

        "missing_cell_rate":
            float(
                profile["missing_cell_rate"]
            ),

        "missing_features":
            int(
                profile.get(
                    "missing_features",
                    quality["missing_features"]
                )
            ),

        "complete_rows":
            int(
                profile.get(
                    "complete_rows",
                    quality["complete_rows"]
                )
            ),

        # ----------------------------------------------------
        # Data quality
        # ----------------------------------------------------

        "duplicate_rows":
            int(
                profile["duplicate_rows"]
            ),

        "duplicate_row_rate":
            float(
                profile.get(
                    "duplicate_row_rate",
                    quality["duplicate_row_rate"]
                )
            ),

        "empty_string_cells":
            int(
                quality["empty_string_cells"]
            ),

        "infinite_numeric_values":
            int(
                quality["infinite_numeric_values"]
            ),

        "fully_null_columns":
            int(
                quality["fully_null_columns"]
            ),

        # ----------------------------------------------------
        # Feature diagnostics
        # ----------------------------------------------------

        "identifier_like_features":
            len(
                identifier_info[
                    "identifier_like_features"
                ]
            ),

        "constant_features":
            len(
                identifier_info[
                    "constant_features"
                ]
            ),

        # ----------------------------------------------------
        # Validation status
        # ----------------------------------------------------

        "schema_valid":
            schema["schema_status"]
            != "MISMATCH",

        "quality_valid":
            bool(
                quality["overall_quality_valid"]
            ),

        "target_valid":
            (
                feature_info["target"]
                in STANDARDIZED_DATASETS[
                    dataset_id
                ].columns
                and
                int(
                    profile[
                        "target_missing_count"
                    ]
                ) == 0
            ),

        # ----------------------------------------------------
        # Notebook hand-off
        # ----------------------------------------------------

        "preparation_status":
            "READY_FOR_NOTEBOOK_02"
    })


# ============================================================
# CREATE SUMMARY DATAFRAME
# ============================================================

DATASET_PREPARATION_SUMMARY_DF = pd.DataFrame(
    PREPARATION_SUMMARY_ROWS
)

display(
    DATASET_PREPARATION_SUMMARY_DF
)


# ============================================================
# FINAL HAND-OFF VALIDATION
# ============================================================

required_status_columns = [
    "schema_valid",
    "quality_valid",
    "target_valid"
]

if not (
    DATASET_PREPARATION_SUMMARY_DF[
        required_status_columns
    ].all().all()
):

    invalid_datasets = (
        DATASET_PREPARATION_SUMMARY_DF.loc[
            ~DATASET_PREPARATION_SUMMARY_DF[
                required_status_columns
            ].all(axis=1),
            "dataset_id"
        ].tolist()
    )

    raise RuntimeError(
        "One or more datasets are not ready for "
        f"Notebook 02: {invalid_datasets}"
    )


# ============================================================
# FINAL REPORT
# ============================================================

print("=" * 100)
print("AIR-LLM — DATASET PREPARATION SUMMARY")
print("=" * 100)

for _, row in (
    DATASET_PREPARATION_SUMMARY_DF.iterrows()
):

    print(
        f"{row['dataset_id']:20s} | "
        f"Rows: {int(row['rows']):8d} | "
        f"Columns: {int(row['columns']):3d} | "
        f"Numeric: {int(row['numeric_features']):2d} | "
        f"Categorical: {int(row['categorical_features']):2d} | "
        f"Target: {row['target']}"
    )

print()
print(
    f"Datasets validated : "
    f"{len(DATASET_PREPARATION_SUMMARY_DF)}"
)

print(
    f"Total observations : "
    f"{int(DATASET_PREPARATION_SUMMARY_DF['rows'].sum()):,}"
)

print(
    f"Total columns      : "
    f"{int(DATASET_PREPARATION_SUMMARY_DF['columns'].sum())}"
)

print()
print(
    "Preparation status : READY_FOR_NOTEBOOK_02"
)

print(
    "No preprocessing or data modification was performed."
)

print("=" * 100)

,dataset_id,dataset_name,task,domain,raw_file,raw_file_format,raw_file_size_bytes,schema_status,reconstruction_method,rows,...,duplicate_row_rate,empty_string_cells,infinite_numeric_values,fully_null_columns,identifier_like_features,constant_features,schema_valid,quality_valid,target_valid,preparation_status
0,adult_income,Adult Income,classification,socioeconomic,adult_income.csv,.csv,3974305,RECONSTRUCTED_HEADERLESS_SCHEMA,CANONICAL_SCHEMA_ASSIGNED,32561,...,0.000737,0,0,0,0,0,True,True,True,READY_FOR_NOTEBOOK_02
1,bank_marketing,Bank Marketing,classification,financial_marketing,bank_marketing.csv,.csv,4610348,OBSERVED_SCHEMA_ACCEPTED,EXISTING_HEADER_PRESERVED,45211,...,0.000000,0,0,0,0,0,True,True,True,READY_FOR_NOTEBOOK_02
2,diabetes_130us,Diabetes 130-US Hospitals,classification,healthcare,diabetes_130us.csv,.csv,16246860,OBSERVED_SCHEMA_ACCEPTED,EXISTING_HEADER_PRESERVED,101766,...,0.000000,0,0,0,3,2,True,True,True,READY_FOR_NOTEBOOK_02


AIR-LLM — DATASET PREPARATION SUMMARY
adult_income         | Rows:    32561 | Columns:  15 | Numeric:  6 | Categorical:  8 | Target: income
bank_marketing       | Rows:    45211 | Columns:  17 | Numeric:  7 | Categorical:  9 | Target: y
diabetes_130us       | Rows:   101766 | Columns:  48 | Numeric: 11 | Categorical: 36 | Target: readmitted

Datasets validated : 3
Total observations : 179,538
Total columns      : 80

Preparation status : READY_FOR_NOTEBOOK_02
No preprocessing or data modification was performed.


In [54]:
# ============================================================
# CELL 01.15 — SAVE DATASET REFERENCE METADATA
# ============================================================
#
# Purpose:
#   Persist the canonical dataset representation produced by
#   Notebook 01 together with immutable schema, metadata,
#   provenance, and cryptographic hashes.
#
# Important:
#   These reference datasets become the controlled input layer
#   for Notebook 02 and all downstream experiments.
#
# No preprocessing, imputation, encoding, scaling, or
# missingness injection is performed here.
# ============================================================

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

# ------------------------------------------------------------
# 1. Reference directories
# ------------------------------------------------------------

REFERENCE_DATASETS_DIR = (
    REFERENCE_DIR / "datasets"
)

REFERENCE_METADATA_DIR = (
    REFERENCE_DIR / "metadata"
)

REFERENCE_SCHEMA_DIR = (
    REFERENCE_DIR / "schemas"
)

for directory in [
    REFERENCE_DATASETS_DIR,
    REFERENCE_METADATA_DIR,
    REFERENCE_SCHEMA_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 2. SHA-256 helper
# ------------------------------------------------------------

def calculate_file_sha256(
    path,
    chunk_size=1024 * 1024
):

    sha256 = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


# ------------------------------------------------------------
# 3. Save reference artifacts
# ------------------------------------------------------------

REFERENCE_ARTIFACTS = []


for dataset_id in DATASETS:

    df = STANDARDIZED_DATASETS[
        dataset_id
    ]

    dataset_config = DATASET_REGISTRY[
        dataset_id
    ]

    feature_info = FEATURE_REGISTRY[
        dataset_id
    ]

    identifier_info = IDENTIFIER_REGISTRY[
        dataset_id
    ]

    raw_path = RAW_FILE_MAP[
        dataset_id
    ]

    # --------------------------------------------------------
    # Reference dataset
    # --------------------------------------------------------

    reference_csv_path = (
        REFERENCE_DATASETS_DIR
        / f"{dataset_id}_reference.csv"
    )

    df.to_csv(
        reference_csv_path,
        index=False
    )

    # --------------------------------------------------------
    # Schema artifact
    # --------------------------------------------------------

    schema_path = (
        REFERENCE_SCHEMA_DIR
        / f"{dataset_id}_schema.json"
    )

    schema_record = {

        "dataset_id":
            dataset_id,

        "dataset_name":
            dataset_config["dataset_name"],

        "task":
            dataset_config["task"],

        "domain":
            dataset_config["domain"],

        "target":
            feature_info["target"],

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "column_order":
            list(df.columns),

        "dtypes": {
            column:
                str(df[column].dtype)
            for column in df.columns
        },

        "numeric_features":
            list(
                feature_info[
                    "numeric_features"
                ]
            ),

        "categorical_features":
            list(
                feature_info[
                    "categorical_features"
                ]
            ),

        "identifier_like_features":
            list(
                identifier_info[
                    "identifier_like_features"
                ]
            ),

        "constant_features":
            list(
                identifier_info[
                    "constant_features"
                ]
            ),

        "missing_counts": {
            column:
                int(df[column].isna().sum())
            for column in df.columns
        },

        "unique_counts": {
            column:
                int(
                    df[column]
                    .nunique(
                        dropna=False
                    )
                )
            for column in df.columns
        },

        "reference_status":
            "CANONICAL_REFERENCE_DATASET"
    }

    with open(
        schema_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            schema_record,
            file,
            indent=4,
            ensure_ascii=False
        )

    # --------------------------------------------------------
    # Complete metadata
    # --------------------------------------------------------

    metadata_path = (
        REFERENCE_METADATA_DIR
        / f"{dataset_id}_metadata.json"
    )

    metadata_record = {

        "dataset_id":
            dataset_id,

        "dataset_name":
            dataset_config["dataset_name"],

        "task":
            dataset_config["task"],

        "domain":
            dataset_config["domain"],

        "target":
            feature_info["target"],

        "raw_file":
            str(raw_path),

        "raw_file_sha256":
            calculate_file_sha256(
                raw_path
            ),

        "reference_file":
            str(reference_csv_path),

        "reference_file_sha256":
            calculate_file_sha256(
                reference_csv_path
            ),

        "rows":
            int(df.shape[0]),

        "columns":
            int(df.shape[1]),

        "column_order":
            list(df.columns),

        "dtypes": {
            column:
                str(df[column].dtype)
            for column in df.columns
        },

        "numeric_features":
            list(
                feature_info[
                    "numeric_features"
                ]
            ),

        "categorical_features":
            list(
                feature_info[
                    "categorical_features"
                ]
            ),

        "identifier_like_features":
            list(
                identifier_info[
                    "identifier_like_features"
                ]
            ),

        "constant_features":
            list(
                identifier_info[
                    "constant_features"
                ]
            ),

        "missing_cells":
            int(
                df.isna()
                .sum()
                .sum()
            ),

        "missing_cell_rate":
            float(
                df.isna()
                .sum()
                .sum()
                /
                (
                    df.shape[0]
                    * df.shape[1]
                )
            ),

        "duplicate_rows":
            int(
                df.duplicated()
                .sum()
            ),

        "prepared_at":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "notebook":
            "01_Dataset_Preparation.ipynb",

        # ----------------------------------------------------
        # Processing-state flags
        # ----------------------------------------------------

        "preprocessing_applied":
            False,

        "missingness_injection_applied":
            False,

        "imputation_applied":
            False,

        "encoding_applied":
            False,

        "scaling_applied":
            False,

        "feature_selection_applied":
            False,

        "row_removal_applied":
            False,

        "column_removal_applied":
            False,

        "reference_status":
            "FINAL_CANONICAL_REFERENCE"
    }

    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            metadata_record,
            file,
            indent=4,
            ensure_ascii=False
        )

    # --------------------------------------------------------
    # Artifact record
    # --------------------------------------------------------

    REFERENCE_ARTIFACTS.append({

        "dataset_id":
            dataset_id,

        "reference_dataset":
            str(reference_csv_path),

        "reference_metadata":
            str(metadata_path),

        "reference_schema":
            str(schema_path),

        "reference_dataset_sha256":
            calculate_file_sha256(
                reference_csv_path
            ),

        "metadata_sha256":
            calculate_file_sha256(
                metadata_path
            ),

        "schema_sha256":
            calculate_file_sha256(
                schema_path
            ),

        "reference_dataset_size_bytes":
            reference_csv_path.stat().st_size,

        "metadata_size_bytes":
            metadata_path.stat().st_size,

        "schema_size_bytes":
            schema_path.stat().st_size
    })


# ------------------------------------------------------------
# 4. Artifact summary
# ------------------------------------------------------------

REFERENCE_ARTIFACTS_DF = pd.DataFrame(
    REFERENCE_ARTIFACTS
)

display(
    REFERENCE_ARTIFACTS_DF
)


# ------------------------------------------------------------
# 5. Artifact existence validation
# ------------------------------------------------------------

for _, row in REFERENCE_ARTIFACTS_DF.iterrows():

    required_paths = [
        Path(row["reference_dataset"]),
        Path(row["reference_metadata"]),
        Path(row["reference_schema"])
    ]

    for path in required_paths:

        if not path.exists():

            raise FileNotFoundError(
                f"Reference artifact was not saved:\n{path}"
            )

        if path.stat().st_size == 0:

            raise ValueError(
                f"Reference artifact is empty:\n{path}"
            )


print("=" * 100)
print("REFERENCE DATASETS AND METADATA SAVED")
print("=" * 100)

print(
    f"Reference datasets : {REFERENCE_DATASETS_DIR}"
)

print(
    f"Metadata            : {REFERENCE_METADATA_DIR}"
)

print(
    f"Schemas             : {REFERENCE_SCHEMA_DIR}"
)

print()
print(
    "Reference artifact validation: PASSED"
)

,dataset_id,reference_dataset,reference_metadata,reference_schema,reference_dataset_sha256,metadata_sha256,schema_sha256,reference_dataset_size_bytes,metadata_size_bytes,schema_size_bytes
0,adult_income,/content/drive/MyDrive/AIR_LLM_Research/data/r...,/content/drive/MyDrive/AIR_LLM_Research/data/r...,/content/drive/MyDrive/AIR_LLM_Research/data/r...,98fbe372dd00d6afcdec811b8bb79925ff35cf6ca47fc8...,8bf84c50bb259acace12a5bca750ab7c4be95476e2bf7f...,e611923a56f3dfcfaf997291827e02c7925c84fe3a356d...,3514344,2397,2334
1,bank_marketing,/content/drive/MyDrive/AIR_LLM_Research/data/r...,/content/drive/MyDrive/AIR_LLM_Research/data/r...,/content/drive/MyDrive/AIR_LLM_Research/data/r...,b44507c26634736d4a0619ec2d7c4b6ff56a2c63cbeb51...,826a57374b790439fb04ae1623127038e7828fb31ed9aa...,25166f97eed52cd7fdf2a330d96be500ffe751a40d1437...,3706094,2358,2283
2,diabetes_130us,/content/drive/MyDrive/AIR_LLM_Research/data/r...,/content/drive/MyDrive/AIR_LLM_Research/data/r...,/content/drive/MyDrive/AIR_LLM_Research/data/r...,89904e16909ac086219844c1cb5f69c64ccf4ff4497696...,c4486a188d55e122ce8f7a163d4fe79fe083810e7e5bb0...,180e743588d1ade917e20611b82a4c5c9285c1a2635fd9...,16246860,5432,7289


REFERENCE DATASETS AND METADATA SAVED
Reference datasets : /content/drive/MyDrive/AIR_LLM_Research/data/reference/datasets
Metadata            : /content/drive/MyDrive/AIR_LLM_Research/data/reference/metadata
Schemas             : /content/drive/MyDrive/AIR_LLM_Research/data/reference/schemas

Reference artifact validation: PASSED


In [55]:
# ============================================================
# CELL 01.15.1 — SAVE ALL PROFILE TABLES
# ============================================================
#
# Purpose:
#   Persist all Notebook 01 diagnostic and profiling tables.
#
# These tables provide:
#   - dataset acquisition provenance
#   - file-format validation
#   - header validation
#   - schema reconciliation
#   - column standardization
#   - target validation
#   - feature-type detection
#   - identifier/constant detection
#   - structural profiling
#   - raw-data quality assessment
#   - final preparation summary
#
# Important:
#   These are audit artifacts only.
#   They do not modify the canonical reference datasets.
# ============================================================

PROFILE_DIR = (
    REFERENCE_DIR / "profiles"
)

PROFILE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 1. Profile tables
# ------------------------------------------------------------

PROFILE_OUTPUTS = {

    "raw_detection":
        RAW_DETECTION_DF,

    "file_format":
        FILE_FORMAT_DF,

    "header_validation":
        HEADER_VALIDATION_DF,

    "dataset_load":
        DATASET_LOAD_DF,

    "schema_reconstruction":
        SCHEMA_RECONCILIATION_DF,

    "column_standardization":
        COLUMN_STANDARDIZATION_DF,

    "target_validation":
        TARGET_VALIDATION_DF,

    "feature_types":
        FEATURE_TYPE_DF,

    "identifier_detection":
        IDENTIFIER_DF,

    "structural_profile":
        STRUCTURAL_PROFILE_DF,

    "raw_quality":
        RAW_QUALITY_DF,

    "preparation_summary":
        DATASET_PREPARATION_SUMMARY_DF
}


# ------------------------------------------------------------
# 2. Save profile tables
# ------------------------------------------------------------

PROFILE_PATHS = {}
PROFILE_VALIDATION_ROWS = []


for name, dataframe in PROFILE_OUTPUTS.items():

    if not isinstance(
        dataframe,
        pd.DataFrame
    ):
        raise TypeError(
            f"Profile '{name}' is not a pandas DataFrame."
        )

    output_path = (
        PROFILE_DIR
        / f"{name}.csv"
    )

    dataframe.to_csv(
        output_path,
        index=False
    )

    # --------------------------------------------------------
    # Validate saved artifact
    # --------------------------------------------------------

    if not output_path.exists():

        raise FileNotFoundError(
            f"Profile table was not saved:\n{output_path}"
        )

    if output_path.stat().st_size == 0:

        raise ValueError(
            f"Profile table is empty:\n{output_path}"
        )

    PROFILE_PATHS[name] = str(
        output_path
    )

    PROFILE_VALIDATION_ROWS.append({

        "profile":
            name,

        "rows":
            int(dataframe.shape[0]),

        "columns":
            int(dataframe.shape[1]),

        "file":
            str(output_path),

        "size_bytes":
            int(output_path.stat().st_size),

        "status":
            "PASS"
    })


# ------------------------------------------------------------
# 3. Profile validation report
# ------------------------------------------------------------

PROFILE_VALIDATION_DF = pd.DataFrame(
    PROFILE_VALIDATION_ROWS
)

display(
    PROFILE_VALIDATION_DF
)


# ------------------------------------------------------------
# 4. Final validation
# ------------------------------------------------------------

assert (
    len(PROFILE_OUTPUTS)
    == len(PROFILE_VALIDATION_DF)
)

assert all(
    PROFILE_VALIDATION_DF["status"]
    == "PASS"
)

assert all(
    PROFILE_VALIDATION_DF["size_bytes"]
    > 0
)


# ------------------------------------------------------------
# 5. Final report
# ------------------------------------------------------------

print("=" * 100)
print("NOTEBOOK 01 — PROFILE TABLES SAVED")
print("=" * 100)

print(
    f"Profile directory : {PROFILE_DIR}"
)

print(
    f"Profile tables    : {len(PROFILE_OUTPUTS)}"
)

print()

for name, path in PROFILE_PATHS.items():

    size = Path(path).stat().st_size

    print(
        f"✓ {name:30s} | "
        f"{size:,} bytes"
    )

print()
print(
    "Profile persistence validation: PASSED"
)

,profile,rows,columns,file,size_bytes,status
0,raw_detection,3,8,/content/drive/MyDrive/AIR_LLM_Research/data/r...,563,PASS
1,file_format,3,6,/content/drive/MyDrive/AIR_LLM_Research/data/r...,233,PASS
2,header_validation,3,6,/content/drive/MyDrive/AIR_LLM_Research/data/r...,1219,PASS
3,dataset_load,3,9,/content/drive/MyDrive/AIR_LLM_Research/data/r...,363,PASS
4,schema_reconstruction,3,9,/content/drive/MyDrive/AIR_LLM_Research/data/r...,408,PASS
5,column_standardization,80,4,/content/drive/MyDrive/AIR_LLM_Research/data/r...,3528,PASS
6,target_validation,3,8,/content/drive/MyDrive/AIR_LLM_Research/data/r...,303,PASS
7,feature_types,80,8,/content/drive/MyDrive/AIR_LLM_Research/data/r...,4996,PASS
8,identifier_detection,80,9,/content/drive/MyDrive/AIR_LLM_Research/data/r...,4823,PASS
9,structural_profile,3,18,/content/drive/MyDrive/AIR_LLM_Research/data/r...,606,PASS


NOTEBOOK 01 — PROFILE TABLES SAVED
Profile directory : /content/drive/MyDrive/AIR_LLM_Research/data/reference/profiles
Profile tables    : 12

✓ raw_detection                  | 563 bytes
✓ file_format                    | 233 bytes
✓ header_validation              | 1,219 bytes
✓ dataset_load                   | 363 bytes
✓ schema_reconstruction          | 408 bytes
✓ column_standardization         | 3,528 bytes
✓ target_validation              | 303 bytes
✓ feature_types                  | 4,996 bytes
✓ identifier_detection           | 4,823 bytes
✓ structural_profile             | 606 bytes
✓ raw_quality                    | 640 bytes
✓ preparation_summary            | 1,309 bytes

Profile persistence validation: PASSED


In [58]:
# ============================================================
# CELL 01.15.2 — SAVE FEATURE REGISTRY
# ============================================================

FEATURE_REGISTRY_PATH = (
    REFERENCE_METADATA_DIR
    / "feature_registry.json"
)

with open(
    FEATURE_REGISTRY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        FEATURE_REGISTRY,
        file,
        indent=4,
        ensure_ascii=False
    )


IDENTIFIER_REGISTRY_PATH = (
    REFERENCE_METADATA_DIR
    / "identifier_registry.json"
)

with open(
    IDENTIFIER_REGISTRY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        IDENTIFIER_REGISTRY,
        file,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# Registry validation
# ------------------------------------------------------------

REGISTRY_VALIDATION_ROWS = []


for dataset_id in DATASETS:

    feature_info = FEATURE_REGISTRY[
        dataset_id
    ]

    identifier_info = IDENTIFIER_REGISTRY[
        dataset_id
    ]

    target = feature_info["target"]

    all_features = feature_info[
        "all_features"
    ]

    numeric_features = feature_info[
        "numeric_features"
    ]

    categorical_features = feature_info[
        "categorical_features"
    ]

    # --------------------------------------------------------
    # Target must be excluded from feature lists
    # --------------------------------------------------------

    if target in all_features:

        raise ValueError(
            f"{dataset_id}: target '{target}' "
            "must not appear in all_features."
        )

    if target in numeric_features:

        raise ValueError(
            f"{dataset_id}: target '{target}' "
            "must not appear in numeric_features."
        )

    if target in categorical_features:

        raise ValueError(
            f"{dataset_id}: target '{target}' "
            "must not appear in categorical_features."
        )

    # --------------------------------------------------------
    # Numeric and categorical features must not overlap
    # --------------------------------------------------------

    overlapping_features = (
        set(numeric_features)
        & set(categorical_features)
    )

    if overlapping_features:

        raise ValueError(
            f"{dataset_id}: feature(s) appear in both "
            f"numeric and categorical registries: "
            f"{sorted(overlapping_features)}"
        )

    # --------------------------------------------------------
    # All non-target features must be classified
    # --------------------------------------------------------

    classified_features = (
        set(numeric_features)
        | set(categorical_features)
    )

    if classified_features != set(
        all_features
    ):

        missing_from_registry = (
            set(all_features)
            - classified_features
        )

        unexpected_in_registry = (
            classified_features
            - set(all_features)
        )

        raise ValueError(
            f"{dataset_id}: feature registry mismatch.\n"
            f"Missing from registry: "
            f"{sorted(missing_from_registry)}\n"
            f"Unexpected in registry: "
            f"{sorted(unexpected_in_registry)}"
        )

    # --------------------------------------------------------
    # Validate identifier / constant registry references
    # --------------------------------------------------------

    identifier_features = identifier_info[
        "identifier_like_features"
    ]

    constant_features = identifier_info[
        "constant_features"
    ]

    invalid_identifiers = (
        set(identifier_features)
        - set(all_features)
    )

    invalid_constants = (
        set(constant_features)
        - set(all_features)
    )

    if invalid_identifiers:

        raise ValueError(
            f"{dataset_id}: identifier registry contains "
            f"unknown features: {sorted(invalid_identifiers)}"
        )

    if invalid_constants:

        raise ValueError(
            f"{dataset_id}: constant registry contains "
            f"unknown features: {sorted(invalid_constants)}"
        )

    # --------------------------------------------------------
    # Validation record
    # --------------------------------------------------------

    REGISTRY_VALIDATION_ROWS.append({

        "dataset_id":
            dataset_id,

        "target":
            target,

        "all_features":
            len(all_features),

        "numeric_features":
            len(numeric_features),

        "categorical_features":
            len(categorical_features),

        "identifier_like_features":
            len(identifier_features),

        "constant_features":
            len(constant_features),

        "registry_valid":
            True
    })


REGISTRY_VALIDATION_DF = pd.DataFrame(
    REGISTRY_VALIDATION_ROWS
)


# ------------------------------------------------------------
# Save validation report
# ------------------------------------------------------------

REGISTRY_VALIDATION_PATH = (
    REFERENCE_METADATA_DIR
    / "registry_validation.csv"
)

REGISTRY_VALIDATION_DF.to_csv(
    REGISTRY_VALIDATION_PATH,
    index=False
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("=" * 100)
print("FEATURE AND IDENTIFIER REGISTRIES SAVED")
print("=" * 100)

print(
    f"Feature registry saved:\n"
    f"{FEATURE_REGISTRY_PATH}"
)

print(
    f"Identifier registry saved:\n"
    f"{IDENTIFIER_REGISTRY_PATH}"
)

print(
    f"Registry validation saved:\n"
    f"{REGISTRY_VALIDATION_PATH}"
)

display(
    REGISTRY_VALIDATION_DF
)

print()
print(
    "Registry validation: PASSED"
)

FEATURE AND IDENTIFIER REGISTRIES SAVED
Feature registry saved:
/content/drive/MyDrive/AIR_LLM_Research/data/reference/metadata/feature_registry.json
Identifier registry saved:
/content/drive/MyDrive/AIR_LLM_Research/data/reference/metadata/identifier_registry.json
Registry validation saved:
/content/drive/MyDrive/AIR_LLM_Research/data/reference/metadata/registry_validation.csv


,dataset_id,target,all_features,numeric_features,categorical_features,identifier_like_features,constant_features,registry_valid
0,adult_income,income,14,6,8,0,0,True
1,bank_marketing,y,16,7,9,0,0,True
2,diabetes_130us,readmitted,47,11,36,3,2,True



Registry validation: PASSED


In [59]:
# ============================================================
# CELL 01.15.3 — SAVE NOTEBOOK 01 MANIFEST
# ============================================================

NOTEBOOK_01_MANIFEST = {

    "notebook":
        "01_Dataset_Preparation.ipynb",

    "status":
        "COMPLETED",

    "created_at":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "project_root":
        str(PROJECT_ROOT),

    "configuration_version":
        CONFIGURATION_LOCK.get(
            "configuration_version"
        ),

    "configuration_status":
        CONFIGURATION_LOCK.get(
            "status"
        ),

    # --------------------------------------------------------
    # Dataset information
    # --------------------------------------------------------

    "datasets":
        list(DATASETS),

    "dataset_count":
        len(DATASETS),

    # --------------------------------------------------------
    # Reference artifact directories
    # --------------------------------------------------------

    "reference_dataset_directory":
        str(
            REFERENCE_DATASETS_DIR
        ),

    "metadata_directory":
        str(
            REFERENCE_METADATA_DIR
        ),

    "schema_directory":
        str(
            REFERENCE_SCHEMA_DIR
        ),

    "profile_directory":
        str(
            PROFILE_DIR
        ),

    # --------------------------------------------------------
    # Core registries
    # --------------------------------------------------------

    "feature_registry":
        str(
            FEATURE_REGISTRY_PATH
        ),

    "identifier_registry":
        str(
            IDENTIFIER_REGISTRY_PATH
        ),

    "registry_validation":
        str(
            REGISTRY_VALIDATION_PATH
        ),

    # --------------------------------------------------------
    # Saved reference artifacts
    # --------------------------------------------------------

    "reference_artifacts":
        [
            {
                "dataset_id":
                    row["dataset_id"],

                "reference_dataset":
                    row["reference_dataset"],

                "reference_metadata":
                    row["reference_metadata"],

                "reference_schema":
                    row["reference_schema"]
            }

            for row in REFERENCE_ARTIFACTS
        ],

    # --------------------------------------------------------
    # Preparation state
    # --------------------------------------------------------

    "data_transformations": {

        "missing_value_imputation":
            False,

        "missingness_generation":
            False,

        "outlier_removal":
            False,

        "encoding":
            False,

        "scaling":
            False,

        "feature_engineering":
            False
    },

    # --------------------------------------------------------
    # Notebook 01 validation status
    # --------------------------------------------------------

    "validation": {

        "raw_file_detection":
            True,

        "file_format_validation":
            True,

        "header_validation":
            True,

        "dataset_loading":
            True,

        "schema_reconciliation":
            True,

        "column_standardization":
            True,

        "target_validation":
            True,

        "feature_type_detection":
            True,

        "identifier_constant_detection":
            True,

        "structural_profiling":
            True,

        "raw_quality_validation":
            True,

        "registry_validation":
            True
    },

    # --------------------------------------------------------
    # Downstream readiness
    # --------------------------------------------------------

    "ready_for_notebook_02":
        True,

    "reference_data_is_raw_semantic_state":
        True,

    "missingness_injection_applied":
        False,

    "imputation_applied":
        False
}


NOTEBOOK_01_MANIFEST_PATH = (
    ARTIFACT_DIR
    / "notebook_01_manifest.json"
)


with open(
    NOTEBOOK_01_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        NOTEBOOK_01_MANIFEST,
        file,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# Final manifest validation
# ------------------------------------------------------------

if not NOTEBOOK_01_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        "Notebook 01 manifest was not created."
    )


print("=" * 100)
print("NOTEBOOK 01 MANIFEST SAVED")
print("=" * 100)

print(
    f"Manifest:\n"
    f"{NOTEBOOK_01_MANIFEST_PATH}"
)

print()
print(
    f"Datasets: {len(DATASETS)}"
)

print(
    "Status: COMPLETED"
)

print(
    "Ready for Notebook 02: TRUE"
)

NOTEBOOK 01 MANIFEST SAVED
Manifest:
/content/drive/MyDrive/AIR_LLM_Research/artifacts/notebook_01_manifest.json

Datasets: 3
Status: COMPLETED
Ready for Notebook 02: TRUE


In [61]:
# ============================================================
# CELL 01.16 — FINAL NOTEBOOK 01 VALIDATION
# ============================================================

VALIDATION_RESULTS = {}


# ------------------------------------------------------------
# Project
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "project_root_exists"
] = (
    PROJECT_ROOT.exists()
    and PROJECT_ROOT.is_dir()
)


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "configuration_locked"
] = (
    CONFIGURATION_LOCK.get(
        "status"
    ) == "LOCKED"
)

VALIDATION_RESULTS[
    "dataset_registry_loaded"
] = (
    len(DATASET_REGISTRY)
    == len(DATASETS)
)


# ------------------------------------------------------------
# Raw files
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "all_raw_files_found"
] = (
    RAW_DETECTION_DF["raw_file_found"]
    .all()
)


VALIDATION_RESULTS[
    "all_formats_supported"
] = (
    FILE_FORMAT_DF["supported"]
    .all()
)


# ------------------------------------------------------------
# Dataset loading
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "all_datasets_loaded"
] = (
    len(RAW_DATASETS)
    == len(DATASETS)
)


VALIDATION_RESULTS[
    "all_datasets_nonempty"
] = all(
    not df.empty
    for df in RAW_DATASETS.values()
)


# ------------------------------------------------------------
# Schema
# ------------------------------------------------------------

# Cell 01.7 produces "schema_status",
# not "schema_match".
schema_status_values = (
    set(
        SCHEMA_RECONCILIATION_DF[
            "schema_status"
        ]
        .astype(str)
        .tolist()
    )
)

allowed_schema_statuses = {
    "MATCH",
    "RECONSTRUCTED_HEADERLESS_SCHEMA",
    "OBSERVED_SCHEMA_ACCEPTED"
}

VALIDATION_RESULTS[
    "schema_validation_passed"
] = (
    len(SCHEMA_RECONCILIATION_DF)
    == len(DATASETS)
    and schema_status_values.issubset(
        allowed_schema_statuses
    )
)


VALIDATION_RESULTS[
    "all_dataset_column_counts_valid"
] = all(
    (
        int(
            DATASET_SCHEMAS[
                dataset_id
            ]["column_count"]
        )
        ==
        len(
            STANDARDIZED_DATASETS[
                dataset_id
            ].columns
        )
    )
    for dataset_id in DATASETS
)


VALIDATION_RESULTS[
    "unique_standardized_columns"
] = all(
    not df.columns.duplicated().any()
    for df
    in STANDARDIZED_DATASETS.values()
)


# ------------------------------------------------------------
# Targets
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "all_targets_exist"
] = (
    TARGET_VALIDATION_DF[
        "target_exists"
    ]
    .all()
)


# ------------------------------------------------------------
# Feature registry
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "feature_registry_complete"
] = (
    set(FEATURE_REGISTRY.keys())
    == set(DATASETS)
)


VALIDATION_RESULTS[
    "identifier_registry_complete"
] = (
    set(IDENTIFIER_REGISTRY.keys())
    == set(DATASETS)
)


# ------------------------------------------------------------
# Feature registry internal consistency
# ------------------------------------------------------------

feature_registry_valid = True

for dataset_id in DATASETS:

    df = STANDARDIZED_DATASETS[
        dataset_id
    ]

    feature_info = FEATURE_REGISTRY[
        dataset_id
    ]

    target = feature_info[
        "target"
    ]

    numeric_features = set(
        feature_info[
            "numeric_features"
        ]
    )

    categorical_features = set(
        feature_info[
            "categorical_features"
        ]
    )

    all_features = set(
        feature_info[
            "all_features"
        ]
    )

    expected_features = (
        set(df.columns)
        - {target}
    )

    if all_features != expected_features:
        feature_registry_valid = False
        break

    if (
        numeric_features
        & categorical_features
    ):
        feature_registry_valid = False
        break

    if (
        numeric_features
        | categorical_features
    ) != all_features:
        feature_registry_valid = False
        break

    if target not in df.columns:
        feature_registry_valid = False
        break


VALIDATION_RESULTS[
    "feature_registry_internally_valid"
] = feature_registry_valid


# ------------------------------------------------------------
# Quality
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "structural_quality_valid"
] = (
    RAW_QUALITY_DF[
        "quality_schema_valid"
    ]
    .all()
)


# ------------------------------------------------------------
# Reference datasets
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "all_reference_datasets_saved"
] = all(
    (
        REFERENCE_DATASETS_DIR
        / f"{dataset_id}_reference.csv"
    ).exists()
    for dataset_id in DATASETS
)


VALIDATION_RESULTS[
    "all_metadata_saved"
] = all(
    (
        REFERENCE_METADATA_DIR
        / f"{dataset_id}_metadata.json"
    ).exists()
    for dataset_id in DATASETS
)


VALIDATION_RESULTS[
    "all_schemas_saved"
] = all(
    (
        REFERENCE_SCHEMA_DIR
        / f"{dataset_id}_schema.json"
    ).exists()
    for dataset_id in DATASETS
)


# ------------------------------------------------------------
# Profiles
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "all_profiles_saved"
] = all(
    (
        PROFILE_DIR
        / f"{name}.csv"
    ).exists()
    for name in PROFILE_OUTPUTS
)


# ------------------------------------------------------------
# Registry files
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "feature_registry_saved"
] = (
    FEATURE_REGISTRY_PATH.exists()
)


VALIDATION_RESULTS[
    "identifier_registry_saved"
] = (
    IDENTIFIER_REGISTRY_PATH.exists()
)


# ------------------------------------------------------------
# Manifest
# ------------------------------------------------------------

VALIDATION_RESULTS[
    "notebook_manifest_saved"
] = (
    NOTEBOOK_01_MANIFEST_PATH.exists()
)


# ------------------------------------------------------------
# Final validation dataframe
# ------------------------------------------------------------

NOTEBOOK_01_VALIDATION_DF = pd.DataFrame(
    [
        {
            "validation": key,
            "status": (
                "PASS"
                if value
                else "FAIL"
            )
        }
        for key, value
        in VALIDATION_RESULTS.items()
    ]
)


display(
    NOTEBOOK_01_VALIDATION_DF
)


# ------------------------------------------------------------
# Failed checks
# ------------------------------------------------------------

failed_checks = [
    key
    for key, value
    in VALIDATION_RESULTS.items()
    if not value
]


# ------------------------------------------------------------
# Final summary values
# ------------------------------------------------------------

dataset_count = len(
    DATASETS
)

reference_count = sum(
    (
        REFERENCE_DATASETS_DIR
        / f"{dataset_id}_reference.csv"
    ).exists()
    for dataset_id in DATASETS
)

schema_count = sum(
    (
        REFERENCE_SCHEMA_DIR
        / f"{dataset_id}_schema.json"
    ).exists()
    for dataset_id in DATASETS
)

profile_count = sum(
    (
        PROFILE_DIR
        / f"{name}.csv"
    ).exists()
    for name in PROFILE_OUTPUTS
)

registry_count = len(
    FEATURE_REGISTRY
)

validation_count = len(
    VALIDATION_RESULTS
)

failed_count = len(
    failed_checks
)


# ------------------------------------------------------------
# Stop if validation failed
# ------------------------------------------------------------

if failed_checks:

    print("=" * 100)
    print(
        "AIR-LLM — NOTEBOOK 01 VALIDATION FAILED"
    )
    print("=" * 100)

    print()

    print(
        "Failed checks:"
    )

    for check in failed_checks:
        print(
            f"  - {check}"
        )

    print()

    raise RuntimeError(
        "Notebook 01 validation failed. "
        "Review the failed checks above."
    )


# ------------------------------------------------------------
# Final publication-ready summary
# ------------------------------------------------------------

print("=" * 100)
print(
    "AIR-LLM — NOTEBOOK 01 FINAL VALIDATION"
)
print("=" * 100)

print(
    f"Datasets processed       : {dataset_count}"
)

print(
    f"Reference datasets saved : {reference_count}"
)

print(
    f"Feature registries       : {registry_count}"
)

print(
    f"Schema files saved       : {schema_count}"
)

print(
    f"Profile tables saved     : {profile_count}"
)

print(
    f"Validation checks        : {validation_count}"
)

print(
    f"Failed checks            : {failed_count}"
)

print(
    f"Configuration status     : "
    f"{CONFIGURATION_LOCK.get('status')}"
)

print()

print(
    "Dataset preparation      : COMPLETE"
)

print(
    "Reference data           : READY"
)

print(
    "Notebook 02 readiness    : YES"
)

print("=" * 100)

print(
    "ALL NOTEBOOK 01 VALIDATIONS PASSED"
)

print(
    "AIR-LLM DATASET PREPARATION IS COMPLETE"
)

print(
    "NOTEBOOK 02 IS AUTHORIZED TO LOAD THE REFERENCE DATA"
)

print("=" * 100)

,validation,status
0,project_root_exists,PASS
1,configuration_locked,PASS
2,dataset_registry_loaded,PASS
3,all_raw_files_found,PASS
4,all_formats_supported,PASS
5,all_datasets_loaded,PASS
6,all_datasets_nonempty,PASS
7,schema_validation_passed,PASS
8,all_dataset_column_counts_valid,PASS
9,unique_standardized_columns,PASS


AIR-LLM — NOTEBOOK 01 FINAL VALIDATION
Datasets processed       : 3
Reference datasets saved : 3
Feature registries       : 3
Schema files saved       : 3
Profile tables saved     : 12
Validation checks        : 22
Failed checks            : 0
Configuration status     : LOCKED

Dataset preparation      : COMPLETE
Reference data           : READY
Notebook 02 readiness    : YES
ALL NOTEBOOK 01 VALIDATIONS PASSED
AIR-LLM DATASET PREPARATION IS COMPLETE
NOTEBOOK 02 IS AUTHORIZED TO LOAD THE REFERENCE DATA


In [62]:
# ============================================================
# CELL 01.16.1 — FINAL DRIVE PERSISTENCE VALIDATION
# ============================================================

REQUIRED_NOTEBOOK_01_FILES = [

    NOTEBOOK_01_MANIFEST_PATH,

    FEATURE_REGISTRY_PATH,

    IDENTIFIER_REGISTRY_PATH
]


# ------------------------------------------------------------
# Dataset-specific artifacts
# ------------------------------------------------------------

for dataset_id in DATASETS:

    REQUIRED_NOTEBOOK_01_FILES.extend([

        REFERENCE_DATASETS_DIR
        / f"{dataset_id}_reference.csv",

        REFERENCE_METADATA_DIR
        / f"{dataset_id}_metadata.json",

        REFERENCE_SCHEMA_DIR
        / f"{dataset_id}_schema.json"
    ])


# ------------------------------------------------------------
# Profile artifacts
# ------------------------------------------------------------

for name in PROFILE_OUTPUTS:

    REQUIRED_NOTEBOOK_01_FILES.append(
        PROFILE_DIR
        / f"{name}.csv"
    )


# ------------------------------------------------------------
# Persistence validation
# ------------------------------------------------------------

missing_persistence_files = [
    str(path)
    for path in REQUIRED_NOTEBOOK_01_FILES
    if not path.exists()
    or not path.is_file()
]


empty_persistence_files = [
    str(path)
    for path in REQUIRED_NOTEBOOK_01_FILES
    if path.exists()
    and path.is_file()
    and path.stat().st_size == 0
]


nonfile_persistence_paths = [
    str(path)
    for path in REQUIRED_NOTEBOOK_01_FILES
    if path.exists()
    and not path.is_file()
]


# ------------------------------------------------------------
# Persistence validation dataframe
# ------------------------------------------------------------

PERSISTENCE_VALIDATION_ROWS = []


for path in REQUIRED_NOTEBOOK_01_FILES:

    exists = path.exists()

    is_file = (
        exists
        and path.is_file()
    )

    file_size = (
        path.stat().st_size
        if is_file
        else 0
    )

    valid = (
        is_file
        and file_size > 0
    )

    PERSISTENCE_VALIDATION_ROWS.append({

        "artifact":
            path.name,

        "path":
            str(path),

        "exists":
            exists,

        "is_file":
            is_file,

        "size_bytes":
            file_size,

        "status":
            "PASS"
            if valid
            else "FAIL"
    })


NOTEBOOK_01_PERSISTENCE_DF = pd.DataFrame(
    PERSISTENCE_VALIDATION_ROWS
)


display(
    NOTEBOOK_01_PERSISTENCE_DF
)


# ------------------------------------------------------------
# Final persistence status
# ------------------------------------------------------------

persistence_validation_passed = (
    len(missing_persistence_files) == 0
    and len(empty_persistence_files) == 0
    and len(nonfile_persistence_paths) == 0
)


if not persistence_validation_passed:

    print("=" * 100)
    print(
        "NOTEBOOK 01 DRIVE PERSISTENCE VALIDATION FAILED"
    )
    print("=" * 100)

    if missing_persistence_files:

        print("\nMissing files:")

        for path in missing_persistence_files:

            print(
                f"  ✗ {path}"
            )

    if empty_persistence_files:

        print("\nEmpty files:")

        for path in empty_persistence_files:

            print(
                f"  ✗ {path}"
            )

    if nonfile_persistence_paths:

        print("\nPaths that are not files:")

        for path in nonfile_persistence_paths:

            print(
                f"  ✗ {path}"
            )

    raise RuntimeError(
        "Notebook 01 artifacts were not completely "
        "persisted to Google Drive."
    )


# ------------------------------------------------------------
# Final successful validation
# ------------------------------------------------------------

required_artifact_count = (
    len(REQUIRED_NOTEBOOK_01_FILES)
)

missing_artifact_count = (
    len(missing_persistence_files)
)

empty_artifact_count = (
    len(empty_persistence_files)
)

nonfile_artifact_count = (
    len(nonfile_persistence_paths)
)

total_size_bytes = sum(
    path.stat().st_size
    for path in REQUIRED_NOTEBOOK_01_FILES
    if path.exists()
    and path.is_file()
)

total_size_mb = (
    total_size_bytes
    / 1024**2
)


print("=" * 100)
print(
    "AIR-LLM — NOTEBOOK 01 DRIVE PERSISTENCE"
)
print("=" * 100)

print(
    f"Required artifacts : "
    f"{required_artifact_count}"
)

print(
    f"Missing artifacts  : "
    f"{missing_artifact_count}"
)

print(
    f"Empty artifacts    : "
    f"{empty_artifact_count}"
)

print(
    f"Non-file paths     : "
    f"{nonfile_artifact_count}"
)

print(
    f"Total persisted    : "
    f"{total_size_mb:.3f} MB"
)

print()

print(
    "GOOGLE DRIVE PERSISTENCE VALIDATION PASSED"
)

print(
    "NOTEBOOK 01 ARTIFACTS ARE SAFELY PERSISTED"
)

print(
    "NOTEBOOK 02 MAY LOAD DATA ONLY FROM "
    "THE PERSISTED REFERENCE ARTIFACTS"
)

print("=" * 100)

,artifact,path,exists,is_file,size_bytes,status
0,notebook_01_manifest.json,/content/drive/MyDrive/AIR_LLM_Research/artifa...,True,True,3389,PASS
1,feature_registry.json,/content/drive/MyDrive/AIR_LLM_Research/data/r...,True,True,5010,PASS
2,identifier_registry.json,/content/drive/MyDrive/AIR_LLM_Research/data/r...,True,True,483,PASS
3,adult_income_reference.csv,/content/drive/MyDrive/AIR_LLM_Research/data/r...,True,True,3514344,PASS
4,adult_income_metadata.json,/content/drive/MyDrive/AIR_LLM_Research/data/r...,True,True,2397,PASS
5,adult_income_schema.json,/content/drive/MyDrive/AIR_LLM_Research/data/r...,True,True,2334,PASS
6,bank_marketing_reference.csv,/content/drive/MyDrive/AIR_LLM_Research/data/r...,True,True,3706094,PASS
7,bank_marketing_metadata.json,/content/drive/MyDrive/AIR_LLM_Research/data/r...,True,True,2358,PASS
8,bank_marketing_schema.json,/content/drive/MyDrive/AIR_LLM_Research/data/r...,True,True,2283,PASS
9,diabetes_130us_reference.csv,/content/drive/MyDrive/AIR_LLM_Research/data/r...,True,True,16246860,PASS


AIR-LLM — NOTEBOOK 01 DRIVE PERSISTENCE
Required artifacts : 24
Missing artifacts  : 0
Empty artifacts    : 0
Non-file paths     : 0
Total persisted    : 22.428 MB

GOOGLE DRIVE PERSISTENCE VALIDATION PASSED
NOTEBOOK 01 ARTIFACTS ARE SAFELY PERSISTED
NOTEBOOK 02 MAY LOAD DATA ONLY FROM THE PERSISTED REFERENCE ARTIFACTS


In [63]:
# ============================================================
# CELL 01.16.2 — FINAL NOTEBOOK STATUS
# ============================================================

print()
print("=" * 100)
print("AIR-LLM — NOTEBOOK 01 COMPLETE")
print("=" * 100)


# ------------------------------------------------------------
# PROJECT
# ------------------------------------------------------------

print("\nPROJECT")
print("-" * 100)

print(
    f"Project              : "
    f"{MASTER_CONFIG['project']['name']}"
)

print(
    f"Project Code         : "
    f"{MASTER_CONFIG['project']['code']}"
)

print(
    f"Configuration        : "
    f"{MASTER_CONFIG['project']['configuration_version']}"
)

print(
    f"Project Root         : "
    f"{PROJECT_ROOT}"
)


# ------------------------------------------------------------
# DATASETS
# ------------------------------------------------------------

print("\nDATASETS")
print("-" * 100)

for dataset_id in DATASETS:

    metadata = DATASET_REGISTRY[
        dataset_id
    ]

    profile = (
        STRUCTURAL_PROFILE_DF.loc[
            STRUCTURAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ].iloc[0]
    )

    target = FEATURE_REGISTRY[
        dataset_id
    ]["target"]

    print(
        f"{dataset_id:20s} | "
        f"{metadata['dataset_name']:30s} | "
        f"Rows: {int(profile['rows']):8d} | "
        f"Columns: {int(profile['columns']):3d} | "
        f"Target: {target}"
    )


# ------------------------------------------------------------
# DATASET STRUCTURE
# ------------------------------------------------------------

print("\nDATASET STRUCTURE")
print("-" * 100)

for dataset_id in DATASETS:

    feature_info = FEATURE_REGISTRY[
        dataset_id
    ]

    identifier_info = IDENTIFIER_REGISTRY[
        dataset_id
    ]

    print(
        f"{dataset_id:20s} | "
        f"Numerical: "
        f"{len(feature_info['numeric_features']):2d} | "
        f"Categorical: "
        f"{len(feature_info['categorical_features']):2d} | "
        f"Identifiers: "
        f"{len(identifier_info['identifier_like_features']):2d} | "
        f"Constants: "
        f"{len(identifier_info['constant_features']):2d}"
    )


# ------------------------------------------------------------
# REFERENCE DATA
# ------------------------------------------------------------

print("\nREFERENCE DATA")
print("-" * 100)

print(
    f"Reference directory   : "
    f"{REFERENCE_DIR}"
)

print(
    f"Datasets saved        : "
    f"{len(DATASETS)}"
)

print(
    f"Metadata files        : "
    f"{len(DATASETS)}"
)

print(
    f"Schema files          : "
    f"{len(DATASETS)}"
)

print(
    f"Profile tables        : "
    f"{len(PROFILE_OUTPUTS)}"
)

print(
    f"Feature registry      : "
    f"{FEATURE_REGISTRY_PATH}"
)

print(
    f"Identifier registry   : "
    f"{IDENTIFIER_REGISTRY_PATH}"
)


# ------------------------------------------------------------
# DRIVE PERSISTENCE
# ------------------------------------------------------------

print("\nDRIVE PERSISTENCE")
print("-" * 100)

persistence_status = (
    "PASSED"
    if persistence_validation_passed
    else "FAILED"
)

print(
    f"Persistence status    : "
    f"{persistence_status}"
)

print(
    f"Required artifacts    : "
    f"{required_artifact_count}"
)

print(
    f"Missing artifacts     : "
    f"{missing_artifact_count}"
)

print(
    f"Empty artifacts       : "
    f"{empty_artifact_count}"
)

print(
    f"Persisted size        : "
    f"{total_size_mb:.3f} MB"
)


# ------------------------------------------------------------
# TRANSFORMATIONS
# ------------------------------------------------------------

print("\nTRANSFORMATIONS")
print("-" * 100)

print(
    "Missing-value imputation : NO"
)

print(
    "Missingness generation   : NO"
)

print(
    "Outlier removal          : NO"
)

print(
    "Encoding                 : NO"
)

print(
    "Scaling                  : NO"
)

print(
    "Feature engineering      : NO"
)


# ------------------------------------------------------------
# DATA INTEGRITY
# ------------------------------------------------------------

print("\nDATA INTEGRITY")
print("-" * 100)

print(
    "Raw observations preserved       : YES"
)

print(
    "Adult Income header reconstructed: YES"
)

print(
    "Original missing values preserved: YES"
)

print(
    "Duplicate observations preserved : YES"
)

print(
    "Identifier-like features removed : NO"
)

print(
    "Constant features removed        : NO"
)


# ------------------------------------------------------------
# RESEARCH PIPELINE STATUS
# ------------------------------------------------------------

print("\nRESEARCH PIPELINE STATUS")
print("-" * 100)

print(
    "Notebook 01 — Dataset Preparation : COMPLETE"
)

print(
    "Reference datasets               : READY"
)

print(
    "Dataset schemas                  : READY"
)

print(
    "Feature registry                 : READY"
)

print(
    "Identifier registry              : READY"
)

print(
    "Profile tables                   : READY"
)

print(
    "Drive persistence                : VERIFIED"
)


# ------------------------------------------------------------
# NEXT NOTEBOOK
# ------------------------------------------------------------

print("\nNEXT NOTEBOOK")
print("-" * 100)

print(
    "Notebook 02 — Data Preprocessing and Splitting"
)

print(
    "Input source:"
)

print(
    f"  {REFERENCE_DATASETS_DIR}"
)


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print()
print("=" * 100)
print(
    "ALL NOTEBOOK 01 VALIDATIONS PASSED"
)
print(
    "AIR-LLM DATASET PREPARATION IS COMPLETE"
)
print(
    "REFERENCE DATA AND METADATA ARE PERSISTED"
)
print(
    "NOTEBOOK 02 IS AUTHORIZED TO BEGIN"
)
print("=" * 100)


AIR-LLM — NOTEBOOK 01 COMPLETE

PROJECT
----------------------------------------------------------------------------------------------------
Project              : AIR-LLM Research Project
Project Code         : AIR-LLM
Configuration        : CONFIG-v1
Project Root         : /content/drive/MyDrive/AIR_LLM_Research

DATASETS
----------------------------------------------------------------------------------------------------
adult_income         | Adult Income                   | Rows:    32561 | Columns:  15 | Target: income
bank_marketing       | Bank Marketing                 | Rows:    45211 | Columns:  17 | Target: y
diabetes_130us       | Diabetes 130-US Hospitals      | Rows:   101766 | Columns:  48 | Target: readmitted

DATASET STRUCTURE
----------------------------------------------------------------------------------------------------
adult_income         | Numerical:  6 | Categorical:  8 | Identifiers:  0 | Constants:  0
bank_marketing       | Numerical:  7 | Categorical:  9 